# Hybrid Work and Job–Home Networks in Spain
## Formal intensity-gradient analysis with partial correlations and contextual-inference extensions, v4.4 (2026-07-24)

> **Public-release setup.** Set the `HYBRID_WORK_PROJECT_ROOT` environment variable to the project directory, or launch this notebook from that directory. The configuration cell uses `pathlib` throughout and documents the expected input and output folders.

This notebook uses all six workplace-recurrence layers and organises the empirical analysis around a continuous hybrid-work intensity gradient.

The revised sample strategy separates four tasks that were previously combined:

1. **National intensity-layer comparisons**
   - all six layer networks;
   - pairwise similarity;
   - national structural gradients and monthly stability.

2. **District-month outcome models**
   - the primary district intensity measure uses all job–home flows, including within-district flows;
   - each outcome uses its own maximum available sample;
   - partner coverage is the observed number of external partners, with external flow controlled in the models;
   - standardised rarefied coverage remains a robustness measure.

3. **Period descriptive maps**
   - period metrics are recalculated from the full 2022–2024 network;
   - maps do not inherit the strict common complete-case regression sample;
   - structural zeros are retained for partner coverage and partner diversity;
   - mean external distance remains undefined where no external relation exists.

4. **Context models**
   - settlement urbanity and existing onsite network position are linked to each outcome independently;
   - each model uses all observations available for its own outcome;
   - model-sample marginal effects remain separate from descriptive map inputs.

The integrated GHSL workflow is retained. Monthly processing caches from the earlier intensity-gradient notebook remain reusable.


## 1. Dependencies

In [ ]:
# Run once if the environment does not already contain these packages.
%pip install -q pandas pyarrow numpy scipy statsmodels patsy matplotlib geopandas pyogrio shapely rasterio requests

## 2. Configuration

In [ ]:
import os
from pathlib import Path

# ---------------------------------------------------------------------
# Project inputs
# ---------------------------------------------------------------------
# Public-release path configuration.
# Set HYBRID_WORK_PROJECT_ROOT to an absolute project directory, or run the
# notebook from the project root. For example:
# PROJECT_ROOT = Path("/path/to/hybrid-work-network-project")
PROJECT_ROOT = Path(
    os.environ.get("HYBRID_WORK_PROJECT_ROOT", Path.cwd())
).expanduser().resolve()
STAGE0_LONG_DIR = PROJECT_ROOT / "02_monthly_network_layers" / "long"

BASEMAP_DIR = PROJECT_ROOT / "basemaps"
REFERENCE_DIR = PROJECT_ROOT / "03_reference"
DISTRICT_CENTROID_PREFERRED = "zonificacion_distritos_centroides.shp"
DISTANCE_CRS = "EPSG:3035"
DISTANCE_METHOD = "official_district_centroids_epsg3035"


# ---------------------------------------------------------------------
# Integrated GHSL settlement profile
# ---------------------------------------------------------------------
DISTRICT_POLYGON_PREFERRED = "zonificacion_distritos.shp"

# The integrated workflow reproduces the earlier national GHS-SMOD
# settlement-profile calculation. It first reuses a validated profile
# from the legacy output location. When no valid profile exists, it can
# download/crop the official rasters and calculate the district profile.
GHS_PROFILE_RUN_TAG = "v1_1"
GHS_OUTPUT_DIR = (
    PROJECT_ROOT
    / "05_analysis"
    / f"06_ghs_smod_all_spain_districts_{GHS_PROFILE_RUN_TAG}"
)
GHS_TABLE_DIR = GHS_OUTPUT_DIR / "tables"
GHS_LOG_DIR = GHS_OUTPUT_DIR / "logs"

GHSL_DIR = REFERENCE_DIR / "ghsl_r2023a"
GHS_SPAIN_RASTER_DIR = GHSL_DIR / "spain_clips"
GHS_SMOD_SPAIN_RASTER = (
    GHS_SPAIN_RASTER_DIR
    / "GHS_SMOD_E2020_R2023A_1km_Spain_extent.tif"
)
GHS_POP_SPAIN_RASTER = (
    GHS_SPAIN_RASTER_DIR
    / "GHS_POP_E2020_R2023A_1km_Spain_extent.tif"
)

GHSL_RELEASE = "R2023A"
GHSL_EPOCH = 2020
GHSL_RESOLUTION_METRES = 1000

GHS_SMOD_URL_CANDIDATES = [
    (
        "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/"
        "GHS_SMOD_GLOBE_R2023A/"
        "GHS_SMOD_E2020_GLOBE_R2023A_54009_1000/"
        "V2-0/GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V2_0.zip"
    ),
    (
        "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/"
        "GHS_SMOD_GLOBE_R2023A/"
        "GHS_SMOD_E2020_GLOBE_R2023A_54009_1000/"
        "V1-0/GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0.zip"
    ),
]
GHS_POP_URL_CANDIDATES = [
    (
        "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/"
        "GHS_POP_GLOBE_R2023A/"
        "GHS_POP_E2020_GLOBE_R2023A_54009_1000/"
        "V1-0/GHS_POP_E2020_GLOBE_R2023A_54009_1000_V1_0.zip"
    ),
]

# Reuse existing results by default. Set GHS_FORCE_REBUILD=True only
# when the profile itself must be recomputed.
GHS_FORCE_REBUILD = False
GHS_BUILD_IF_MISSING = True
GHS_DOWNLOAD_IF_MISSING = True
GHS_RESUME_FROM_CHECKPOINT = True
GHS_CHECKPOINT_EVERY_N_DISTRICTS = 50
GHS_PROGRESS_EVERY_N_DISTRICTS = 25
GHS_MAX_DISTRICTS = None
GHS_MIN_OVERLAP_FRACTION = 1e-9
GHS_SPAIN_CLIP_PADDING_CELLS = 2
GHS_DELETE_GLOBAL_SOURCES_AFTER_CLIP = True
GHS_REQUEST_TIMEOUT_SECONDS = 180
GHS_DOWNLOAD_CHUNK_BYTES = 1024 * 1024

# ---------------------------------------------------------------------
# Article outputs. A new stage directory avoids collisions with the
# earlier binary Hybrid–Onsite analysis.
# ---------------------------------------------------------------------
ANALYSIS_ROOT = PROJECT_ROOT / "05_analysis"
ARTICLE_ROOT = ANALYSIS_ROOT / "13_hybrid_network_article"

# Change these two values together when starting a new full run.
RUN_VERSION = "v4_4"
RUN_DATE = "20260724"
RUN_TAG = f"{RUN_VERSION}_{RUN_DATE}"
STAGE_DIR = (
    ARTICLE_ROOT
    / f"01_intensity_gradient_analysis_{RUN_TAG}"
)

CACHE_DIR = STAGE_DIR / "monthly_cache"
TABLE_MAIN_DIR = STAGE_DIR / "tables_main"
TABLE_APPENDIX_DIR = STAGE_DIR / "tables_appendix"
FIGURE_MAIN_DIR = STAGE_DIR / "figures_main"
FIGURE_APPENDIX_DIR = STAGE_DIR / "figures_appendix"
LOG_DIR = STAGE_DIR / "logs"
MAP_INPUT_DIR = STAGE_DIR / "map_inputs_for_notebook_02"

for folder in [
    STAGE_DIR, CACHE_DIR, TABLE_MAIN_DIR, TABLE_APPENDIX_DIR,
    FIGURE_MAIN_DIR, FIGURE_APPENDIX_DIR, LOG_DIR, MAP_INPUT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Six hybrid-work intensity layers
# ---------------------------------------------------------------------
RECURRENCE_BAND_ORDER = ["1-2", "3-4", "5-7", "8-10", "11-13", "14"]
RECURRENCE_MIDPOINT = {
    "1-2": 1.5,
    "3-4": 3.5,
    "5-7": 6.0,
    "8-10": 9.0,
    "11-13": 12.0,
    "14": 14.0,
}
RECURRENCE_INTENSITY = {
    band: (14.0 - midpoint) / (14.0 - 1.5)
    for band, midpoint in RECURRENCE_MIDPOINT.items()
}
HIGH_ATTENDANCE_BANDS = ["8-10", "11-13", "14"]
REPRESENTATIVE_MAP_BANDS = ["1-2", "5-7", "14"]

# The older binary definitions are retained only for optional appendix
# sensitivity outputs and do not structure the main analysis.
NETWORK_COMPARISONS = {
    "adjacent_layers": {
        "hybrid_bands": ["5-7"],
        "onsite_bands": ["8-10"],
        "label": "5–7 days versus 8–10 days",
        "main": False,
    }
}
PREMIUM_SPECS = []
MAIN_SPEC = "adjacent_layers"

# ---------------------------------------------------------------------
# Quality control and analytical samples
# ---------------------------------------------------------------------
REQUIRED_STAGE0_COLUMNS = {
    "month", "origin", "destination", "recurrence_band", "layer_weight"
}
MIN_NODE_COVERAGE_RATIO = 0.80
MIN_EDGE_COVERAGE_RATIO = 0.80
MIN_FLOW_COVERAGE_RATIO = 0.50
MIN_DISTANCE_COVERAGE_SHARE = 0.90

EXCLUDE_MONTHS = set()
FORCE_INCLUDE_MONTHS = set()
FORCE_REPROCESS_MONTHS = set()

MIN_EXTERNAL_PARTNERS = 2
MIN_EXTERNAL_FLOW = 0.0
# Descriptive period maps use every district with at least one valid
# observation. Model-specific samples are selected independently.
MIN_VALID_MONTHS_PERIOD = 1
MIN_VALID_MONTHS_CONTEXT = 1
NODE_RAREFACTION_DRAW_CAP = 100_000
NETWORK_RAREFACTION_DRAW_CAP = 100_000

# A fixed national draw makes partner coverage comparable across districts,
# months and roles. The main draw is selected from the 10th percentile of
# eligible district-month external flows. Two nearby quantiles are exported
# as appendix sensitivity checks.
PARTNER_COVERAGE_DRAW_QUANTILE = 0.10
PARTNER_COVERAGE_SENSITIVITY_QUANTILES = [0.05, 0.15]

# ---------------------------------------------------------------------
# Maximum-available-sample definitions
# ---------------------------------------------------------------------
PRIMARY_INTENSITY_COLUMN = "hybrid_intensity_all_flows"
ROBUSTNESS_INTENSITY_COLUMN = "hybrid_intensity_external"

# Main partner coverage is the observed number of external partners.
# The model controls external flow. Rarefied coverage remains available
# at the 5th, 10th and 15th percentile common draws.
PRIMARY_PARTNER_COVERAGE_COLUMN = "partner_coverage"
PRIMARY_PARTNER_COVERAGE_DEFINITION = "observed_external_partner_count"

# Structural zero policy:
# - no external relation -> partner coverage = 0
# - no external relation -> partner diversity = 0
# - no external relation -> mean external distance remains undefined
INCLUDE_STRUCTURAL_ZEROS_COVERAGE = True
INCLUDE_STRUCTURAL_ZEROS_DIVERSITY = True
DISTANCE_WITHOUT_EXTERNAL_RELATION = float("nan")

# ---------------------------------------------------------------------
# Model settings
# ---------------------------------------------------------------------
FE_DEMEAN_TOLERANCE = 1e-10
FE_DEMEAN_MAX_ITER = 500
INTENSITY_EFFECT_INCREMENT = 0.10
CONTEXT_PERCENTILES = [0.20, 0.50, 0.80]

# ---------------------------------------------------------------------
# Runtime and output controls
# ---------------------------------------------------------------------
REUSE_MONTHLY_CACHE = True
RUN_PERIOD_SPECTRAL_CENTRALITY = True
REQUIRE_GHS_PROFILE = True
# Minimum share of network districts that must match the GHSL profile.
# The check prevents silent sample loss caused by incompatible district IDs.
MIN_GHS_MATCH_SHARE = 0.90
PARQUET_COMPRESSION = "zstd"
RANDOM_SEED = 20260722
FIGURE_DPI = 400
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True

ROLE_LABELS = {
    "residential": "Residential role",
    "employment": "Employment role",
}

PUBLICATION_LABELS = {
    "external_flow_share": "Inter-district flow share",
    "mean_external_distance_km": "Mean external distance (km)",
    "rarefied_expected_edge_count": "Network connections at a common flow scale",
    "edge_hhi": "Flow concentration (HHI)",
    "partner_coverage": "Partner coverage",
    "partner_coverage_rarefied_q10": "Standardised partner coverage",
    "effective_partner_diversity": "Partner diversity",
    "hybrid_intensity_external": "External-flow hybrid-work intensity",
    "hybrid_intensity_all_flows": "Hybrid-work intensity",
    "settlement_urbanity": "Settlement urbanity",
    "onsite_network_position": "Existing onsite network position",
}

MAIN_FIGURE_STEMS = [
    "figure_1_pairwise_similarity_across_intensity_layers",
    "figure_2_network_structure_across_intensity_layers",
    "figure_3_monthly_network_structure_across_intensity_layers",
    "figure_4_intensity_and_district_network_outcomes",
    "figure_5_urbanity_moderation_of_intensity_outcomes",
    "figure_6_network_position_moderation_of_intensity_outcomes",
]

print("Stage-0 inputs:", STAGE0_LONG_DIR)
print("Intensity-gradient outputs:", STAGE_DIR)
print("Hybrid-work intensity scores:", RECURRENCE_INTENSITY)


## 3. Imports, reproducibility signature, and general utilities

In [ ]:
import hashlib
import json
import math
import re
import shutil
import time
import unicodedata
import warnings
import zipfile
from datetime import datetime

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patsy
import pyarrow.parquet as pq
import rasterio
import requests
import statsmodels.api as sm
import statsmodels.formula.api as smf

from itertools import combinations

from rasterio.features import geometry_mask
from rasterio.windows import Window, from_bounds
from requests.adapters import HTTPAdapter
from shapely.geometry import box, mapping
from shapely.prepared import prep
from urllib3.util.retry import Retry

from scipy import sparse, stats
from scipy.spatial.distance import jensenshannon

pd.set_option("display.max_columns", 320)
pd.set_option("display.max_rows", 240)
pd.set_option("display.width", 280)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
})


def canonical_id(value):
    if value is None or pd.isna(value):
        return None
    text = unicodedata.normalize("NFKC", str(value).strip())
    text = re.sub(r"\.0$", "", text)
    text = re.sub(r"\s+", "", text)
    return text.upper() if text else None


def province_code_from_district(value):
    value = canonical_id(value)
    match = re.match(r"^(\d{2})", value or "")
    return match.group(1) if match else "NA"


def atomic_to_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(path)


def atomic_to_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(temporary, index=False, compression=PARQUET_COMPRESSION)
    temporary.replace(path)


def atomic_write_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def save_figure(figure, stem, directory=FIGURE_MAIN_DIR):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    suffixes = []
    if SAVE_PNG:
        suffixes.append(".png")
    if SAVE_PDF:
        suffixes.append(".pdf")
    if SAVE_SVG:
        suffixes.append(".svg")
    for suffix in suffixes:
        path = directory / f"{stem}{suffix}"
        figure.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
        print("Saved:", path)
    plt.show()
    plt.close(figure)


def require_columns(frame, required, frame_name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise KeyError(
            f"{frame_name} lacks required fields: {missing}. "
            f"Available fields: {frame.columns.tolist()}"
        )


def weighted_quantile(values, weights, quantile):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[valid], weights[valid]
    if len(values) == 0:
        return np.nan
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cumulative = np.cumsum(weights)
    index = np.searchsorted(cumulative, quantile * cumulative[-1], side="left")
    return float(values[min(index, len(values) - 1)])


def entropy_metrics(weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights[np.isfinite(weights) & (weights > 0)]
    if len(weights) == 0:
        return {
            "entropy": np.nan,
            "effective_count": 0.0,
            "normalized_entropy": np.nan,
            "hhi": np.nan,
            "largest_share": np.nan,
            "positive_count": 0,
        }
    probabilities = weights / weights.sum()
    entropy = -np.sum(probabilities * np.log(probabilities))
    normalized = entropy / np.log(len(probabilities)) if len(probabilities) > 1 else 0.0
    return {
        "entropy": float(entropy),
        "effective_count": float(np.exp(entropy)),
        "normalized_entropy": float(normalized),
        "hhi": float(np.sum(probabilities ** 2)),
        "largest_share": float(probabilities.max()),
        "positive_count": int(len(probabilities)),
    }


def expected_occupied_count(probabilities, draws):
    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = probabilities[np.isfinite(probabilities) & (probabilities > 0)]
    if len(probabilities) == 0 or draws <= 0:
        return np.nan
    probabilities = probabilities / probabilities.sum()
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        occupied_probability = -np.expm1(draws * np.log1p(-probabilities))
    return float(np.nansum(occupied_probability))


INTENSITY_COLOURS = {
    band: plt.cm.viridis(value)
    for band, value in zip(
        RECURRENCE_BAND_ORDER,
        np.linspace(0.08, 0.92, len(RECURRENCE_BAND_ORDER)),
    )
}

SIGNATURE_PAYLOAD = {
    "run_tag": RUN_TAG,
    "recurrence_band_order": RECURRENCE_BAND_ORDER,
    "recurrence_intensity": RECURRENCE_INTENSITY,
    "high_attendance_bands": HIGH_ATTENDANCE_BANDS,
    "partner_coverage_draw_quantile": PARTNER_COVERAGE_DRAW_QUANTILE,
    "partner_coverage_sensitivity_quantiles": PARTNER_COVERAGE_SENSITIVITY_QUANTILES,
    "min_node_coverage_ratio": MIN_NODE_COVERAGE_RATIO,
    "min_edge_coverage_ratio": MIN_EDGE_COVERAGE_RATIO,
    "min_flow_coverage_ratio": MIN_FLOW_COVERAGE_RATIO,
    "min_distance_coverage_share": MIN_DISTANCE_COVERAGE_SHARE,
    "min_external_partners": MIN_EXTERNAL_PARTNERS,
    "min_external_flow": MIN_EXTERNAL_FLOW,
    "node_rarefaction_draw_cap": NODE_RAREFACTION_DRAW_CAP,
    "network_rarefaction_draw_cap": NETWORK_RAREFACTION_DRAW_CAP,
    "distance_crs": DISTANCE_CRS,
    "distance_method": DISTANCE_METHOD,
}
ANALYSIS_SIGNATURE = hashlib.sha256(
    json.dumps(SIGNATURE_PAYLOAD, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print("Analysis signature:", ANALYSIS_SIGNATURE)


## 4. Discover and audit the Stage-0 monthly long-edge files

In [ ]:
LONG_PATTERN = re.compile(r"network_layer_edges_long_(\d{4})_(\d{2})\.parquet$")


def month_from_path(path):
    match = LONG_PATTERN.search(Path(path).name)
    if not match:
        return None
    return f"{match.group(1)}-{match.group(2)}"


if not STAGE0_LONG_DIR.exists():
    raise FileNotFoundError(f"Stage-0 long-edge directory does not exist: {STAGE0_LONG_DIR}")

long_edge_files = []
for path in sorted(STAGE0_LONG_DIR.glob("network_layer_edges_long_*.parquet")):
    month = month_from_path(path)
    if month:
        long_edge_files.append((month, path))

if not long_edge_files:
    raise FileNotFoundError(f"No Stage-0 long-edge files were found in {STAGE0_LONG_DIR}")

qc_rows = []
all_network_ids = set()
for month, path in long_edge_files:
    schema = set(pq.ParquetFile(path).schema.names)
    missing = REQUIRED_STAGE0_COLUMNS - schema
    if missing:
        raise KeyError(f"{path.name} lacks required Stage-0 columns: {sorted(missing)}")

    frame = pd.read_parquet(
        path,
        columns=["origin", "destination", "recurrence_band", "layer_weight"],
    )
    frame["origin"] = frame["origin"].map(canonical_id)
    frame["destination"] = frame["destination"].map(canonical_id)
    frame["recurrence_band"] = frame["recurrence_band"].astype(str)
    frame["layer_weight"] = pd.to_numeric(frame["layer_weight"], errors="coerce").fillna(0.0)
    frame = frame.loc[
        frame["origin"].notna()
        & frame["destination"].notna()
        & frame["recurrence_band"].isin(RECURRENCE_BAND_ORDER)
        & (frame["layer_weight"] > 0)
    ].copy()

    all_network_ids.update(frame["origin"].unique())
    all_network_ids.update(frame["destination"].unique())
    pooled_edge_count = frame[["origin", "destination"]].drop_duplicates().shape[0]
    active_nodes = set(frame["origin"]) | set(frame["destination"])
    qc_rows.append({
        "month": month,
        "path": str(path),
        "file_size_mb": path.stat().st_size / 1024**2,
        "positive_layer_edge_rows": int(len(frame)),
        "pooled_positive_edge_count": int(pooled_edge_count),
        "active_node_count": int(len(active_nodes)),
        "total_layer_flow": float(frame["layer_weight"].sum()),
        "observed_layer_count": int(frame["recurrence_band"].nunique()),
        "all_six_layers_present": bool(
            set(RECURRENCE_BAND_ORDER).issubset(set(frame["recurrence_band"]))
        ),
    })

month_qc = pd.DataFrame(qc_rows).sort_values("month").reset_index(drop=True)
for column in ["active_node_count", "pooled_positive_edge_count", "total_layer_flow"]:
    median = month_qc[column].median()
    month_qc[f"{column}_ratio_to_median"] = month_qc[column] / median

month_qc["automatic_valid_month"] = (
    month_qc["all_six_layers_present"]
    & (month_qc["active_node_count_ratio_to_median"] >= MIN_NODE_COVERAGE_RATIO)
    & (month_qc["pooled_positive_edge_count_ratio_to_median"] >= MIN_EDGE_COVERAGE_RATIO)
    & (month_qc["total_layer_flow_ratio_to_median"] >= MIN_FLOW_COVERAGE_RATIO)
)
month_qc["valid_month"] = month_qc["automatic_valid_month"]
month_qc.loc[month_qc["month"].isin(EXCLUDE_MONTHS), "valid_month"] = False
month_qc.loc[month_qc["month"].isin(FORCE_INCLUDE_MONTHS), "valid_month"] = True


def exclusion_reason(row):
    if row["month"] in EXCLUDE_MONTHS:
        return "manual_exclusion"
    if row["month"] in FORCE_INCLUDE_MONTHS:
        return "manual_inclusion"
    reasons = []
    if not row["all_six_layers_present"]:
        reasons.append("missing_recurrence_layer")
    if row["active_node_count_ratio_to_median"] < MIN_NODE_COVERAGE_RATIO:
        reasons.append("low_node_coverage")
    if row["pooled_positive_edge_count_ratio_to_median"] < MIN_EDGE_COVERAGE_RATIO:
        reasons.append("low_edge_coverage")
    if row["total_layer_flow_ratio_to_median"] < MIN_FLOW_COVERAGE_RATIO:
        reasons.append("low_flow_coverage")
    return ";".join(reasons) if reasons else "included"


month_qc["sample_decision_reason"] = month_qc.apply(exclusion_reason, axis=1)
VALID_MONTHS = month_qc.loc[month_qc["valid_month"], "month"].tolist()
VALID_FILE_BY_MONTH = {
    row.month: Path(row.path)
    for row in month_qc.loc[month_qc["valid_month"]].itertuples(index=False)
}

atomic_to_csv(month_qc, LOG_DIR / "stage0_month_sample_audit.csv")
print(f"Discovered {len(month_qc)} Stage-0 months; retained {len(VALID_MONTHS)}.")
display(month_qc)

## 5. Build the official district-centroid lookup


In [ ]:
def resolve_centroid_path():
    preferred_candidates = [
        BASEMAP_DIR / DISTRICT_CENTROID_PREFERRED,
        REFERENCE_DIR / DISTRICT_CENTROID_PREFERRED,
    ]
    for candidate in preferred_candidates:
        if candidate.exists():
            return candidate
    candidates = []
    for directory in [BASEMAP_DIR, REFERENCE_DIR]:
        if directory.exists():
            candidates.extend(sorted(directory.glob("zonificacion_distritos_centroides*.shp")))
    if not candidates:
        raise FileNotFoundError(
            "No official district-centroid shapefile was found in basemaps or 03_reference."
        )
    return candidates[0]


def validate_shapefile_components(shp_path):
    shp_path = Path(shp_path)
    missing = [
        shp_path.with_suffix(suffix)
        for suffix in [".shp", ".shx", ".dbf", ".prj"]
        if not shp_path.with_suffix(suffix).exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing required shapefile components:\n" + "\n".join(str(path) for path in missing)
        )


def score_spatial_id_fields(frame, target_ids):
    target_ids = set(target_ids)
    rows = []
    geometry_name = frame.geometry.name
    for column in frame.columns:
        if column == geometry_name:
            continue
        values = frame[column].map(canonical_id)
        unique_values = set(values.dropna().unique())
        intersection = unique_values & target_ids
        rows.append({
            "field": column,
            "match_count": len(intersection),
            "target_match_share": len(intersection) / len(target_ids) if target_ids else np.nan,
            "field_unique_match_share": len(intersection) / len(unique_values) if unique_values else np.nan,
            "field_unique_values": len(unique_values),
        })
    return pd.DataFrame(rows).sort_values(
        ["target_match_share", "field_unique_match_share", "match_count"],
        ascending=[False, False, False],
    ).reset_index(drop=True)


centroid_path = resolve_centroid_path()
validate_shapefile_components(centroid_path)
centroids = gpd.read_file(centroid_path)
if centroids.crs is None:
    raise ValueError(f"Centroid shapefile has no CRS: {centroid_path}")

id_scores = score_spatial_id_fields(centroids, all_network_ids)
if id_scores.empty or int(id_scores.iloc[0]["match_count"]) == 0:
    raise RuntimeError("No centroid attribute field matches the Stage-0 district IDs.")
centroid_id_field = id_scores.iloc[0]["field"]
centroids["district_id"] = centroids[centroid_id_field].map(canonical_id)
centroids = centroids.loc[centroids["district_id"].notna()].copy()
centroids = centroids.to_crs(DISTANCE_CRS)
if not centroids.geom_type.isin(["Point", "MultiPoint"]).all():
    centroids["geometry"] = centroids.geometry.representative_point()
centroids = centroids.drop_duplicates("district_id")
centroids["x"] = centroids.geometry.x
centroids["y"] = centroids.geometry.y

coordinate_lookup = centroids.set_index("district_id")[["x", "y"]]
x_lookup = coordinate_lookup["x"]
y_lookup = coordinate_lookup["y"]
coordinate_match_share = len(set(coordinate_lookup.index) & all_network_ids) / len(all_network_ids)

atomic_to_csv(id_scores, LOG_DIR / "centroid_id_field_scores.csv")
atomic_to_parquet(
    centroids[["district_id", "x", "y"]],
    TABLE_APPENDIX_DIR / "district_centroid_lookup_epsg3035.parquet",
)
print("Centroid shapefile:", centroid_path)
print("Matched ID field:", centroid_id_field)
print(f"Network-node coordinate match: {coordinate_match_share:.2%}")
display(id_scores.head(10))


## 6. Build or load the district-level GHS-SMOD urbanity profile

This section integrates the earlier national GHS-SMOD workflow into Notebook 01. It first reuses the profile stored under `05_analysis/06_ghs_smod_all_spain_districts_v*/tables/district_settlement_profiles.parquet`. When no validated profile exists, it calculates the district profiles from the official GHS-SMOD R2023A and GHS-POP R2023A rasters for epoch 2020. The continuous urbanity index is the population-weighted mean of equally spaced ordinal scores assigned to the seven inhabited GHS-SMOD classes.


In [ ]:
# ---------------------------------------------------------------------
# GHS-SMOD classifications and integrated storage paths
# ---------------------------------------------------------------------
SMOD_CLASS_NAMES = {
    30: "Urban centre",
    23: "Dense urban cluster",
    22: "Semi-dense urban cluster",
    21: "Suburban or peri-urban",
    13: "Rural cluster",
    12: "Low-density rural",
    11: "Very-low-density rural",
    10: "Water",
}
SMOD_DETAILED_SLUGS = {
    30: "urban_centre",
    23: "dense_urban_cluster",
    22: "semi_dense_urban_cluster",
    21: "suburban_periurban",
    13: "rural_cluster",
    12: "low_density_rural",
    11: "very_low_density_rural",
}
SMOD_FOUR_CLASS = {
    30: "Urban centre",
    23: "Urban cluster / town",
    22: "Urban cluster / town",
    21: "Suburban / peri-urban",
    13: "Rural",
    12: "Rural",
    11: "Rural",
}
FOUR_CLASS_SLUGS = {
    "Urban centre": "urban_centre",
    "Urban cluster / town": "urban_cluster_town",
    "Suburban / peri-urban": "suburban_periurban",
    "Rural": "rural",
}
FOUR_CLASS_ORDER = [
    "Urban centre",
    "Urban cluster / town",
    "Suburban / peri-urban",
    "Rural",
]
SMOD_ORDINAL_SCORE = {
    11: 0.0,
    12: 1.0 / 6.0,
    13: 2.0 / 6.0,
    21: 3.0 / 6.0,
    22: 4.0 / 6.0,
    23: 5.0 / 6.0,
    30: 1.0,
}
INHABITED_SMOD_CODES = set(SMOD_ORDINAL_SCORE)

for folder in [
    GHS_OUTPUT_DIR,
    GHS_TABLE_DIR,
    GHS_LOG_DIR,
    GHSL_DIR,
    GHS_SPAIN_RASTER_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

GHS_PROFILE_PATH = GHS_TABLE_DIR / "district_settlement_profiles.parquet"
GHS_PROFILE_CSV_PATH = GHS_TABLE_DIR / "district_settlement_profiles.csv"
GHS_CHECKPOINT_PATH = GHS_TABLE_DIR / "district_settlement_profiles_checkpoint.parquet"
GHS_ERROR_PATH = GHS_LOG_DIR / "district_processing_errors.csv"


def valid_ghs_profile(path):
    path = Path(path)
    required = {
        "district_id",
        "urban_rural_position_index_0_1",
        "population_weighted_total",
    }
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        columns = set(pq.ParquetFile(path).schema.names)
    except Exception:
        return False
    return required.issubset(columns)


def discover_legacy_ghs_profile():
    """Reproduce the exact search convention used by the older Stage-3 notebook."""
    preferred = GHS_PROFILE_PATH
    if valid_ghs_profile(preferred) and not GHS_FORCE_REBUILD:
        return preferred

    candidates = sorted(
        ANALYSIS_ROOT.glob(
            "06_ghs_smod_all_spain_districts_v*/tables/"
            "district_settlement_profiles.parquet"
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    candidates = [path for path in candidates if valid_ghs_profile(path)]
    if candidates and not GHS_FORCE_REBUILD:
        return candidates[0]
    return None


def make_ghsl_http_session():
    retry = Retry(
        total=6,
        connect=6,
        read=6,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET", "HEAD"),
    )
    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "Academic-GHSL-Spain-District-Analysis/1.0"
        )
    })
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


GHSL_HTTP = make_ghsl_http_session()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def download_with_progress(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    with GHSL_HTTP.get(
        url,
        stream=True,
        timeout=GHS_REQUEST_TIMEOUT_SECONDS,
    ) as response:
        response.raise_for_status()
        total_bytes = int(response.headers.get("content-length", 0))
        downloaded = 0
        started = time.time()
        with open(temporary, "wb") as handle:
            for chunk in response.iter_content(
                chunk_size=GHS_DOWNLOAD_CHUNK_BYTES
            ):
                if not chunk:
                    continue
                handle.write(chunk)
                downloaded += len(chunk)
                if total_bytes:
                    elapsed = max(time.time() - started, 1e-6)
                    print(
                        f"\r{destination.name}: "
                        f"{downloaded / total_bytes:6.1%} | "
                        f"{downloaded / elapsed / 1024**2:6.2f} MB/s",
                        end="",
                    )
    temporary.replace(destination)
    print()
    return destination


def obtain_ghsl_archive(label, candidate_urls):
    errors = []
    for url in candidate_urls:
        destination = GHSL_DIR / url.rsplit("/", 1)[-1]
        if destination.exists() and zipfile.is_zipfile(destination):
            print(f"{label}: using cached archive {destination}")
            return destination, url
        if destination.exists():
            destination.unlink(missing_ok=True)
        if not GHS_DOWNLOAD_IF_MISSING:
            errors.append(f"Missing locally: {destination}")
            continue
        try:
            print(f"{label}: downloading from\n{url}")
            download_with_progress(url, destination)
            if not zipfile.is_zipfile(destination):
                raise zipfile.BadZipFile(
                    f"Downloaded file is not a valid ZIP: {destination}"
                )
            return destination, url
        except Exception as error:
            destination.unlink(missing_ok=True)
            errors.append(f"{url}: {error}")
    raise RuntimeError(
        f"Could not obtain {label}:\n" + "\n".join(errors)
    )


def extract_first_geotiff(archive_path, extraction_folder):
    archive_path = Path(archive_path)
    extraction_folder = Path(extraction_folder)
    extraction_folder.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        tif_members = [
            member for member in archive.namelist()
            if member.lower().endswith((".tif", ".tiff"))
            and not member.endswith("/")
        ]
        if not tif_members:
            raise FileNotFoundError(
                f"No GeoTIFF was found in {archive_path}"
            )
        member = sorted(
            tif_members,
            key=lambda name: (
                0 if "54009_1000" in name else 1,
                len(name),
                name,
            ),
        )[0]
        destination = extraction_folder / Path(member).name
        if not destination.exists() or destination.stat().st_size == 0:
            with archive.open(member) as source, open(destination, "wb") as target:
                shutil.copyfileobj(source, target)
    return destination, member


def raster_is_valid(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        with rasterio.open(path) as source:
            return (
                source.crs is not None
                and source.width > 0
                and source.height > 0
                and source.count >= 1
            )
    except Exception:
        return False


def crop_raster_to_district_extent(source_path, destination_path, district_frame):
    source_path = Path(source_path)
    destination_path = Path(destination_path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination_path.with_suffix(
        destination_path.suffix + ".tmp"
    )

    with rasterio.open(source_path) as source:
        district_bounds = district_frame.to_crs(source.crs).total_bounds
        padding_x = abs(source.res[0]) * GHS_SPAIN_CLIP_PADDING_CELLS
        padding_y = abs(source.res[1]) * GHS_SPAIN_CLIP_PADDING_CELLS
        requested = from_bounds(
            district_bounds[0] - padding_x,
            district_bounds[1] - padding_y,
            district_bounds[2] + padding_x,
            district_bounds[3] + padding_y,
            transform=source.transform,
        ).round_offsets().round_lengths()
        full_window = Window(
            col_off=0,
            row_off=0,
            width=source.width,
            height=source.height,
        )
        window = requested.intersection(full_window)
        data = source.read(window=window)
        profile = source.profile.copy()
        output_height = int(window.height)
        output_width = int(window.width)
        profile.update({
            "height": output_height,
            "width": output_width,
            "transform": source.window_transform(window),
            "compress": "deflate",
            "BIGTIFF": "IF_SAFER",
        })
        # GeoTIFF tile dimensions must be multiples of 16. Preserve
        # tiled writing for the national clips, but fall back to strips
        # for unusually small debugging extents.
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)
        if output_width >= 16 and output_height >= 16:
            block_x = min(512, max(16, (output_width // 16) * 16))
            block_y = min(512, max(16, (output_height // 16) * 16))
            profile.update({
                "tiled": True,
                "blockxsize": block_x,
                "blockysize": block_y,
            })
        else:
            profile["tiled"] = False
        with rasterio.open(temporary, "w", **profile) as target:
            target.write(data)
            target.update_tags(**source.tags())
    temporary.replace(destination_path)
    if not raster_is_valid(destination_path):
        raise RuntimeError(
            f"Spain-only raster failed validation: {destination_path}"
        )
    return destination_path


def prepare_one_spain_raster(
    label,
    candidate_urls,
    extraction_folder,
    destination_path,
    district_frame,
):
    destination_path = Path(destination_path)
    if raster_is_valid(destination_path):
        print(f"{label}: using Spain-only raster {destination_path}")
        return destination_path, [], []

    archive_path, source_url = obtain_ghsl_archive(
        label, candidate_urls
    )
    global_raster, zip_member = extract_first_geotiff(
        archive_path, extraction_folder
    )
    if not raster_is_valid(global_raster):
        raise RuntimeError(
            f"Extracted {label} raster is invalid: {global_raster}"
        )

    crop_raster_to_district_extent(
        global_raster, destination_path, district_frame
    )
    manifest = [{
        "dataset": label,
        "release": GHSL_RELEASE,
        "epoch": GHSL_EPOCH,
        "source_url": source_url,
        "archive_path": str(archive_path),
        "archive_sha256": sha256_file(archive_path),
        "zip_member": zip_member,
        "global_raster": str(global_raster),
        "spain_raster": str(destination_path),
    }]
    cleanup = []
    if GHS_DELETE_GLOBAL_SOURCES_AFTER_CLIP:
        for source_file in [global_raster, archive_path]:
            if Path(source_file).exists():
                size_mb = Path(source_file).stat().st_size / 1024**2
                Path(source_file).unlink()
                cleanup.append({
                    "dataset": label,
                    "removed_path": str(source_file),
                    "released_size_mb": size_mb,
                })
    return destination_path, manifest, cleanup

In [ ]:
def resolve_district_polygon_path():
    preferred = [
        BASEMAP_DIR / DISTRICT_POLYGON_PREFERRED,
        REFERENCE_DIR / DISTRICT_POLYGON_PREFERRED,
    ]
    for candidate in preferred:
        if candidate.exists():
            validate_shapefile_components(candidate)
            return candidate
    candidates = []
    for folder in [BASEMAP_DIR, REFERENCE_DIR]:
        if folder.exists():
            candidates.extend(
                sorted(folder.glob("zonificacion_distritos*.shp"))
            )
    candidates = [
        path for path in candidates
        if "centroid" not in path.stem.lower()
    ]
    if not candidates:
        raise FileNotFoundError(
            "No official MITMA district-polygon shapefile was found."
        )
    validate_shapefile_components(candidates[0])
    return candidates[0]


def candidate_id_variants(series, target_length=None):
    canonical = series.map(canonical_id)
    digits = canonical.map(
        lambda value: re.sub(r"\D", "", value)
        if value is not None else None
    )
    variants = {
        "canonical": canonical,
        "digits_only": digits,
    }
    if target_length is not None:
        variants["canonical_zfill"] = canonical.map(
            lambda value: value.zfill(target_length)
            if value is not None and value.isdigit() else value
        )
        variants["digits_zfill"] = digits.map(
            lambda value: value.zfill(target_length)
            if value is not None and value.isdigit() else value
        )
    return variants


def modal_numeric_length(values):
    lengths = [
        len(value) for value in values
        if value is not None and str(value).isdigit()
    ]
    return int(pd.Series(lengths).mode().iloc[0]) if lengths else None


def match_district_polygons_to_network(district_frame, target_ids):
    target_ids = set(target_ids)
    target_length = modal_numeric_length(target_ids)
    geometry_column = district_frame.geometry.name
    rows = []
    candidate_cache = {}
    for field in district_frame.columns:
        if field == geometry_column:
            continue
        variants = candidate_id_variants(
            district_frame[field],
            target_length=target_length,
        )
        candidate_cache[field] = variants
        for variant_name, values in variants.items():
            unique_values = set(values.dropna().unique())
            matched = unique_values & target_ids
            rows.append({
                "field": field,
                "variant": variant_name,
                "match_count": len(matched),
                "target_match_share": (
                    len(matched) / len(target_ids)
                    if target_ids else np.nan
                ),
                "field_unique_match_share": (
                    len(matched) / len(unique_values)
                    if unique_values else np.nan
                ),
                "field_unique_values": len(unique_values),
            })
    scores = (
        pd.DataFrame(rows)
        .sort_values(
            ["target_match_share", "field_unique_match_share", "match_count"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )
    if scores.empty or int(scores.iloc[0]["match_count"]) == 0:
        raise RuntimeError(
            "No district-polygon field matches the network district IDs."
        )
    best = scores.iloc[0]
    result = district_frame.copy()
    result["district_id"] = candidate_cache[
        best["field"]
    ][best["variant"]]
    result = (
        result.loc[
            result["district_id"].isin(target_ids)
            & result.geometry.notna()
            & ~result.geometry.is_empty
        ]
        .drop_duplicates("district_id")
        .copy()
    )
    if hasattr(result.geometry, "make_valid"):
        result["geometry"] = result.geometry.make_valid()
    else:
        result["geometry"] = result.geometry.buffer(0)
    result = result.loc[
        result.geometry.notna() & ~result.geometry.is_empty
    ].copy()
    return result, scores, best.to_dict()


def rasters_are_aligned(source_a, source_b, tolerance=1e-9):
    return (
        source_a.crs == source_b.crs
        and source_a.width == source_b.width
        and source_a.height == source_b.height
        and np.allclose(
            np.array(source_a.transform)[:6],
            np.array(source_b.transform)[:6],
            atol=tolerance,
            rtol=0,
        )
    )


def safe_four_class_entropy(shares):
    values = np.asarray(shares, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    if len(values) == 0:
        return np.nan, np.nan
    raw = float(-np.sum(values * np.log(values)))
    return raw, float(raw / np.log(4.0))


def classify_four_contexts(four_shares):
    urban_centre = four_shares.get("Urban centre", 0.0)
    urban_cluster = four_shares.get("Urban cluster / town", 0.0)
    suburban = four_shares.get("Suburban / peri-urban", 0.0)
    rural = four_shares.get("Rural", 0.0)
    dominant = max(
        FOUR_CLASS_ORDER,
        key=lambda name: four_shares.get(name, 0.0),
    )
    dominant_share = four_shares.get(dominant, 0.0)
    if urban_centre >= 0.50:
        degree_3 = "City / urban centre"
    elif rural > 0.50:
        degree_3 = "Rural"
    else:
        degree_3 = "Towns and suburbs"
    if urban_centre >= 0.50:
        territorial_4 = "Urban centre"
    elif rural >= 0.50:
        territorial_4 = "Rural"
    elif urban_cluster >= suburban:
        territorial_4 = "Urban cluster / town"
    else:
        territorial_4 = "Suburban / peri-urban"
    return dominant, dominant_share, degree_3, territorial_4


def summarize_one_ghsl_district(
    district_id,
    geometry,
    district_area_km2,
    smod_source,
    pop_source,
):
    requested = from_bounds(
        *geometry.bounds,
        transform=smod_source.transform,
    ).round_offsets().round_lengths()
    full_window = Window(
        col_off=0,
        row_off=0,
        width=smod_source.width,
        height=smod_source.height,
    )
    window = requested.intersection(full_window)
    if window.width <= 0 or window.height <= 0:
        raise RuntimeError(
            "District bounds do not intersect the Spain GHSL rasters."
        )

    smod_array = smod_source.read(
        1, window=window, masked=True
    )
    pop_array = pop_source.read(
        1, window=window, masked=True
    )
    local_transform = smod_source.window_transform(window)
    candidate_mask = geometry_mask(
        [mapping(geometry)],
        out_shape=smod_array.shape,
        transform=local_transform,
        invert=True,
        all_touched=True,
    )
    smod_mask = np.ma.getmaskarray(smod_array)
    candidate_rows, candidate_columns = np.where(
        candidate_mask & ~smod_mask
    )

    prepared_geometry = prep(geometry)
    cell_area = abs(local_transform.a * local_transform.e)
    population_by_code = {
        code: 0.0 for code in INHABITED_SMOD_CODES
    }
    area_by_code = {
        code: 0.0 for code in INHABITED_SMOD_CODES
    }

    touched_cell_count = 0
    inhabited_cell_fragment_count = 0
    positive_population_fragment_count = 0
    inhabited_overlap_area_m2 = 0.0
    water_overlap_area_m2 = 0.0

    for row, column in zip(
        candidate_rows.tolist(),
        candidate_columns.tolist(),
    ):
        smod_code = int(smod_array[row, column])
        if (
            smod_code not in INHABITED_SMOD_CODES
            and smod_code != 10
        ):
            continue

        left = (
            local_transform.c
            + column * local_transform.a
            + row * local_transform.b
        )
        top = (
            local_transform.f
            + column * local_transform.d
            + row * local_transform.e
        )
        right = left + local_transform.a
        bottom = top + local_transform.e
        cell_geometry = box(
            min(left, right),
            min(bottom, top),
            max(left, right),
            max(bottom, top),
        )

        if prepared_geometry.covers(cell_geometry):
            overlap_area = cell_area
        else:
            overlap_area = cell_geometry.intersection(geometry).area
        if overlap_area <= 0:
            continue
        overlap_fraction = overlap_area / cell_area
        if overlap_fraction < GHS_MIN_OVERLAP_FRACTION:
            continue

        touched_cell_count += 1
        if smod_code == 10:
            water_overlap_area_m2 += overlap_area
            continue

        inhabited_cell_fragment_count += 1
        inhabited_overlap_area_m2 += overlap_area
        area_by_code[smod_code] += overlap_area

        pop_masked = np.ma.is_masked(pop_array[row, column])
        pop_value = (
            0.0 if pop_masked
            else float(pop_array[row, column])
        )
        if not np.isfinite(pop_value) or pop_value < 0:
            pop_value = 0.0
        weighted_population = pop_value * overlap_fraction
        population_by_code[smod_code] += weighted_population
        if weighted_population > 0:
            positive_population_fragment_count += 1

    total_population = float(sum(population_by_code.values()))
    total_inhabited_area = float(sum(area_by_code.values()))
    result = {
        "district_id": district_id,
        "district_area_km2": float(district_area_km2),
        "candidate_raster_cells": int(len(candidate_rows)),
        "touched_cell_count": int(touched_cell_count),
        "inhabited_cell_fragment_count": int(
            inhabited_cell_fragment_count
        ),
        "positive_population_fragment_count": int(
            positive_population_fragment_count
        ),
        "inhabited_overlap_area_km2": (
            inhabited_overlap_area_m2 / 1e6
        ),
        "water_overlap_area_km2": (
            water_overlap_area_m2 / 1e6
        ),
        "population_weighted_total": total_population,
    }

    for code in sorted(INHABITED_SMOD_CODES):
        slug = SMOD_DETAILED_SLUGS[code]
        result[f"population_weighted_smod_{code}_{slug}"] = (
            population_by_code[code]
        )
        result[f"population_share_smod_{code}_{slug}"] = (
            population_by_code[code] / total_population
            if total_population > 0 else np.nan
        )
        result[f"area_share_smod_{code}_{slug}"] = (
            area_by_code[code] / total_inhabited_area
            if total_inhabited_area > 0 else np.nan
        )

    four_population = {
        name: 0.0 for name in FOUR_CLASS_ORDER
    }
    four_area = {
        name: 0.0 for name in FOUR_CLASS_ORDER
    }
    for code in INHABITED_SMOD_CODES:
        four_name = SMOD_FOUR_CLASS[code]
        four_population[four_name] += population_by_code[code]
        four_area[four_name] += area_by_code[code]

    four_population_shares = {}
    for name in FOUR_CLASS_ORDER:
        slug = FOUR_CLASS_SLUGS[name]
        population_share = (
            four_population[name] / total_population
            if total_population > 0 else np.nan
        )
        area_share = (
            four_area[name] / total_inhabited_area
            if total_inhabited_area > 0 else np.nan
        )
        four_population_shares[name] = (
            0.0 if not np.isfinite(population_share)
            else float(population_share)
        )
        result[f"population_share_4_{slug}"] = population_share
        result[f"area_share_4_{slug}"] = area_share

    if total_population > 0:
        entropy_raw, entropy_normalized = safe_four_class_entropy(
            [
                four_population_shares[name]
                for name in FOUR_CLASS_ORDER
            ]
        )
        (
            dominant_class,
            dominant_share,
            degree_3,
            territorial_4,
        ) = classify_four_contexts(four_population_shares)
        urban_rural_position = (
            sum(
                population_by_code[code]
                * SMOD_ORDINAL_SCORE[code]
                for code in INHABITED_SMOD_CODES
            )
            / total_population
        )
        mixedness = 1.0 - dominant_share
        four_share_sum = sum(four_population_shares.values())
    else:
        entropy_raw = np.nan
        entropy_normalized = np.nan
        dominant_class = None
        dominant_share = np.nan
        degree_3 = None
        territorial_4 = None
        urban_rural_position = np.nan
        mixedness = np.nan
        four_share_sum = np.nan

    result.update({
        "settlement_entropy_raw_4class": entropy_raw,
        "settlement_entropy_normalized_4class": (
            entropy_normalized
        ),
        "settlement_mixedness_1_minus_dominant_share": mixedness,
        "dominant_four_class": dominant_class,
        "dominant_four_class_population_share": dominant_share,
        "degree_of_urbanisation_3class": degree_3,
        "territorial_position_4class_50pct_rule": territorial_4,
        "urban_rural_position_index_0_1": urban_rural_position,
        "four_class_population_share_sum": four_share_sum,
        "population_available": total_population > 0,
        "quality_status": (
            "PASS"
            if total_population > 0
            and np.isclose(four_share_sum, 1.0, atol=1e-8)
            else "REVIEW"
        ),
    })
    return result


def build_national_ghs_profile():
    if GHS_FORCE_REBUILD:
        for stale_path in [
            GHS_PROFILE_PATH,
            GHS_PROFILE_CSV_PATH,
            GHS_CHECKPOINT_PATH,
            GHS_ERROR_PATH,
        ]:
            Path(stale_path).unlink(missing_ok=True)

    polygon_path = resolve_district_polygon_path()
    districts_raw = gpd.read_file(polygon_path)
    if districts_raw.crs is None:
        raise ValueError(
            f"District polygon shapefile has no CRS: {polygon_path}"
        )
    districts, polygon_id_scores, selected_id = (
        match_district_polygons_to_network(
            districts_raw, all_network_ids
        )
    )
    if districts.empty:
        raise RuntimeError(
            "No district polygons remain after matching network IDs."
        )
    atomic_to_csv(
        polygon_id_scores,
        GHS_LOG_DIR / "district_id_field_scores.csv",
    )
    atomic_to_csv(
        pd.DataFrame([{
            "polygon_path": str(polygon_path),
            "network_node_count": len(all_network_ids),
            "matched_polygon_count": len(districts),
            "selected_id_field": selected_id.get("field"),
            "selected_id_variant": selected_id.get("variant"),
            "target_match_share": selected_id.get(
                "target_match_share"
            ),
        }]),
        GHS_LOG_DIR / "district_input_inventory.csv",
    )

    download_records = []
    cleanup_records = []
    smod_path, manifest, cleanup = prepare_one_spain_raster(
        label="GHS-SMOD",
        candidate_urls=GHS_SMOD_URL_CANDIDATES,
        extraction_folder=GHSL_DIR / "smod_2020",
        destination_path=GHS_SMOD_SPAIN_RASTER,
        district_frame=districts,
    )
    download_records.extend(manifest)
    cleanup_records.extend(cleanup)
    pop_path, manifest, cleanup = prepare_one_spain_raster(
        label="GHS-POP",
        candidate_urls=GHS_POP_URL_CANDIDATES,
        extraction_folder=GHSL_DIR / "pop_2020",
        destination_path=GHS_POP_SPAIN_RASTER,
        district_frame=districts,
    )
    download_records.extend(manifest)
    cleanup_records.extend(cleanup)

    if download_records:
        atomic_to_csv(
            pd.DataFrame(download_records),
            GHS_LOG_DIR / "ghsl_download_and_clip_manifest.csv",
        )
    if cleanup_records:
        atomic_to_csv(
            pd.DataFrame(cleanup_records),
            GHS_LOG_DIR / "global_source_cleanup.csv",
        )

    with rasterio.open(smod_path) as smod_source, rasterio.open(
        pop_path
    ) as pop_source:
        if not rasters_are_aligned(smod_source, pop_source):
            raise RuntimeError(
                "Spain GHS-SMOD and GHS-POP rasters are not aligned."
            )
        districts_ghsl = districts.to_crs(smod_source.crs).copy()
        districts_ghsl["district_area_km2"] = (
            districts_ghsl.geometry.area / 1e6
        )
        atomic_to_csv(
            pd.DataFrame([{
                "smod_path": str(smod_path),
                "pop_path": str(pop_path),
                "crs": str(smod_source.crs),
                "width": smod_source.width,
                "height": smod_source.height,
                "resolution_x": smod_source.res[0],
                "resolution_y": smod_source.res[1],
                "aligned": True,
            }]),
            GHS_LOG_DIR / "spain_raster_validation.csv",
        )

        if (
            GHS_RESUME_FROM_CHECKPOINT
            and GHS_CHECKPOINT_PATH.exists()
            and not GHS_FORCE_REBUILD
        ):
            checkpoint = pd.read_parquet(GHS_CHECKPOINT_PATH)
            checkpoint["district_id"] = checkpoint[
                "district_id"
            ].map(canonical_id)
        else:
            checkpoint = pd.DataFrame()
        completed_ids = set(
            checkpoint.get(
                "district_id", pd.Series(dtype=object)
            ).dropna()
        )
        remaining = districts_ghsl.loc[
            ~districts_ghsl["district_id"].isin(completed_ids)
        ].copy()
        if GHS_MAX_DISTRICTS is not None:
            remaining = remaining.head(
                int(GHS_MAX_DISTRICTS)
            ).copy()

        new_results = []
        processing_errors = []
        started = time.time()
        for index, row in enumerate(
            remaining.itertuples(index=False),
            start=1,
        ):
            try:
                result = summarize_one_ghsl_district(
                    district_id=row.district_id,
                    geometry=row.geometry,
                    district_area_km2=row.district_area_km2,
                    smod_source=smod_source,
                    pop_source=pop_source,
                )
                new_results.append(result)
            except Exception as error:
                processing_errors.append({
                    "district_id": row.district_id,
                    "error_type": type(error).__name__,
                    "error_message": str(error),
                })

            if (
                index % GHS_PROGRESS_EVERY_N_DISTRICTS == 0
                or index == len(remaining)
            ):
                print(
                    f"GHS profile: {index:,}/{len(remaining):,} "
                    f"districts processed; errors={len(processing_errors):,}"
                )
            if (
                index % GHS_CHECKPOINT_EVERY_N_DISTRICTS == 0
                or index == len(remaining)
            ):
                combined = pd.concat(
                    [checkpoint, pd.DataFrame(new_results)],
                    ignore_index=True,
                )
                if not combined.empty:
                    combined["district_id"] = combined[
                        "district_id"
                    ].map(canonical_id)
                    combined = combined.drop_duplicates(
                        "district_id", keep="last"
                    )
                    atomic_to_parquet(
                        combined, GHS_CHECKPOINT_PATH
                    )
                if processing_errors:
                    atomic_to_csv(
                        pd.DataFrame(processing_errors),
                        GHS_ERROR_PATH,
                    )

    if GHS_CHECKPOINT_PATH.exists():
        profiles = pd.read_parquet(GHS_CHECKPOINT_PATH)
    else:
        profiles = pd.DataFrame(new_results)
    profiles["district_id"] = profiles["district_id"].map(
        canonical_id
    )
    profiles = (
        profiles.drop_duplicates("district_id", keep="last")
        .sort_values("district_id")
        .reset_index(drop=True)
    )
    atomic_to_parquet(profiles, GHS_PROFILE_PATH)
    atomic_to_csv(profiles, GHS_PROFILE_CSV_PATH)

    expected_ids = set(districts["district_id"])
    profile_ids = set(profiles["district_id"])
    completion = pd.DataFrame([{
        "expected_districts": len(expected_ids),
        "profile_rows": len(profiles),
        "unique_profile_districts": profiles[
            "district_id"
        ].nunique(),
        "missing_districts": len(expected_ids - profile_ids),
        "unexpected_districts": len(profile_ids - expected_ids),
        "pass_rows": int(
            (profiles["quality_status"] == "PASS").sum()
        ),
        "review_rows": int(
            (profiles["quality_status"] != "PASS").sum()
        ),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }])
    atomic_to_csv(
        completion,
        GHS_LOG_DIR / "national_completion_summary.csv",
    )
    print("Saved integrated GHS profile:", GHS_PROFILE_PATH)
    display(completion)
    return GHS_PROFILE_PATH

In [ ]:
ghs_path = discover_legacy_ghs_profile()
ghs_profile_origin = "existing_profile"

if ghs_path is None:
    if not GHS_BUILD_IF_MISSING:
        raise FileNotFoundError(
            "No validated GHS district profile was found and "
            "GHS_BUILD_IF_MISSING is False."
        )
    ghs_path = build_national_ghs_profile()
    ghs_profile_origin = "integrated_build"

ghs = pd.read_parquet(ghs_path)
require_columns(
    ghs,
    [
        "district_id",
        "urban_rural_position_index_0_1",
        "population_weighted_total",
    ],
    "GHS district profile",
)
optional_columns = [
    column for column in [
        "settlement_entropy_normalized_4class",
        "settlement_mixedness_1_minus_dominant_share",
        "population_share_4_urban_centre",
        "population_share_4_urban_cluster_town",
        "population_share_4_suburban_periurban",
        "population_share_4_rural",
        "dominant_four_class",
        "degree_of_urbanisation_3class",
        "territorial_position_4class_50pct_rule",
        "quality_status",
    ]
    if column in ghs.columns
]
ghs = ghs[[
    "district_id",
    "urban_rural_position_index_0_1",
    "population_weighted_total",
    *optional_columns,
]].copy()
ghs["district_id"] = ghs["district_id"].map(canonical_id)
ghs = (
    ghs.loc[ghs["district_id"].notna()]
    .drop_duplicates("district_id", keep="last")
    .rename(columns={
        "urban_rural_position_index_0_1": "settlement_urbanity",
    })
    .reset_index(drop=True)
)
ghs["settlement_urbanity"] = pd.to_numeric(
    ghs["settlement_urbanity"], errors="coerce"
)
ghs["population_weighted_total"] = pd.to_numeric(
    ghs["population_weighted_total"], errors="coerce"
)
ghs["log_population"] = np.log1p(
    ghs["population_weighted_total"].clip(lower=0)
)
ghs_profile = ghs.copy()

required_ghs_profile_columns = [
    "district_id",
    "settlement_urbanity",
    "population_weighted_total",
    "log_population",
]
require_columns(
    ghs_profile,
    required_ghs_profile_columns,
    "Standardised GHS district profile",
)
if REQUIRE_GHS_PROFILE and ghs_profile.empty:
    raise RuntimeError(
        "The standardised GHS district profile is empty."
    )
if not ghs_profile["district_id"].is_unique:
    raise ValueError(
        "`ghs_profile` must contain one row per district."
    )

network_district_count = len(all_network_ids)
matched_network_districts = len(
    set(ghs_profile["district_id"].dropna())
    & set(all_network_ids)
)
ghs_network_match_share = (
    matched_network_districts / network_district_count
    if network_district_count else np.nan
)
ghs_profile_audit = pd.DataFrame([{
    "ghs_source_path": str(ghs_path),
    "ghs_profile_origin": ghs_profile_origin,
    "ghsl_release": GHSL_RELEASE,
    "ghsl_epoch": GHSL_EPOCH,
    "ghsl_resolution_metres": GHSL_RESOLUTION_METRES,
    "urbanity_definition": (
        "Population-weighted mean of ordinal scores assigned to "
        "seven inhabited GHS-SMOD classes"
    ),
    "ghs_profile_rows": int(len(ghs_profile)),
    "ghs_profile_unique_districts": int(
        ghs_profile["district_id"].nunique()
    ),
    "network_unique_districts": int(network_district_count),
    "matched_network_districts": int(matched_network_districts),
    "network_district_match_share": float(
        ghs_network_match_share
    ),
    "minimum_required_match_share": float(
        MIN_GHS_MATCH_SHARE
    ),
}])
atomic_to_csv(
    ghs_profile_audit,
    LOG_DIR / "ghs_profile_audit.csv",
)
atomic_to_parquet(
    ghs_profile,
    TABLE_APPENDIX_DIR
    / "district_ghs_profile_used_in_analysis.parquet",
)

print("GHS profile:", ghs_path)
print("Profile origin:", ghs_profile_origin)
print(
    "Standardised GHS profile:",
    f"{ghs_profile['district_id'].nunique():,} districts; "
    f"network match = {ghs_network_match_share:.2%}"
)
if (
    REQUIRE_GHS_PROFILE
    and np.isfinite(ghs_network_match_share)
    and ghs_network_match_share < MIN_GHS_MATCH_SHARE
):
    raise RuntimeError(
        "The GHSL district-profile match is below the configured "
        f"threshold: {ghs_network_match_share:.2%} < "
        f"{MIN_GHS_MATCH_SHARE:.2%}."
    )
display(ghs_profile_audit)
display(
    ghs_profile[
        [
            "district_id",
            "settlement_urbanity",
            "population_weighted_total",
        ]
    ].head()
)

## 7. Edge construction, distance, network metrics, similarity, and node reach

In [ ]:
def attach_centroid_distances(edges):
    working = edges.copy()
    if "distance_km" in working.columns:
        stage0_distance = pd.to_numeric(working["distance_km"], errors="coerce")
    else:
        stage0_distance = pd.Series(np.nan, index=working.index, dtype=float)

    origin_x = working["origin"].map(x_lookup)
    origin_y = working["origin"].map(y_lookup)
    destination_x = working["destination"].map(x_lookup)
    destination_y = working["destination"].map(y_lookup)
    external = working["origin"] != working["destination"]
    matched = origin_x.notna() & origin_y.notna() & destination_x.notna() & destination_y.notna()

    computed = pd.Series(np.nan, index=working.index, dtype=float)
    valid = external & matched
    computed.loc[valid] = (
        np.hypot(
            destination_x.loc[valid] - origin_x.loc[valid],
            destination_y.loc[valid] - origin_y.loc[valid],
        ) / 1000.0
    )
    working["distance_km"] = computed
    fallback = external & working["distance_km"].isna() & stage0_distance.notna()
    working.loc[fallback, "distance_km"] = stage0_distance.loc[fallback]
    working.loc[~external, "distance_km"] = 0.0
    return working


def distance_coverage_summary(edges, weight_column="weight"):
    external = edges.loc[
        (edges["origin"] != edges["destination"])
        & (edges[weight_column] > 0)
    ].copy()
    external_flow = float(external[weight_column].sum())
    covered_flow = float(
        external.loc[external["distance_km"].notna(), weight_column].sum()
    )
    return {
        "external_flow": external_flow,
        "distance_covered_external_flow": covered_flow,
        "distance_coverage_share": covered_flow / external_flow if external_flow > 0 else np.nan,
        "external_edge_count": int(len(external)),
        "valid_distance_edge_count": int(external["distance_km"].notna().sum()),
    }


def read_stage0_month(path, month):
    available = set(pq.ParquetFile(path).schema.names)
    columns = [
        "month", "origin", "destination", "recurrence_band", "layer_weight"
    ]
    if "distance_km" in available:
        columns.append("distance_km")
    frame = pd.read_parquet(path, columns=columns)
    frame["origin"] = frame["origin"].map(canonical_id)
    frame["destination"] = frame["destination"].map(canonical_id)
    frame["recurrence_band"] = frame["recurrence_band"].astype(str)
    frame["layer_weight"] = pd.to_numeric(frame["layer_weight"], errors="coerce").fillna(0.0)
    frame = frame.loc[
        frame["origin"].notna()
        & frame["destination"].notna()
        & frame["recurrence_band"].isin(RECURRENCE_BAND_ORDER)
        & (frame["layer_weight"] > 0)
    ].copy()
    frame["month"] = month
    frame = attach_centroid_distances(frame)
    return (
        frame.groupby(["month", "origin", "destination", "recurrence_band"], as_index=False)
        .agg(layer_weight=("layer_weight", "sum"), distance_km=("distance_km", "first"))
    )


def network_from_bands(month_edges, bands):
    working = month_edges.loc[month_edges["recurrence_band"].isin(bands)].copy()
    if working.empty:
        return pd.DataFrame(columns=["origin", "destination", "weight", "distance_km"])
    return (
        working.groupby(["origin", "destination"], as_index=False)
        .agg(weight=("layer_weight", "sum"), distance_km=("distance_km", "first"))
        .loc[lambda frame: frame["weight"] > 0]
        .reset_index(drop=True)
    )


def weighted_reciprocity(edges):
    external = edges.loc[
        (edges["origin"] != edges["destination"]) & (edges["weight"] > 0),
        ["origin", "destination", "weight"],
    ].copy()
    if external.empty:
        return np.nan
    reverse = external.rename(columns={
        "origin": "destination",
        "destination": "origin",
        "weight": "reverse_weight",
    })
    paired = external.merge(reverse, on=["origin", "destination"], how="left")
    paired["reverse_weight"] = paired["reverse_weight"].fillna(0.0)
    reciprocal_mass = np.minimum(paired["weight"], paired["reverse_weight"]).sum()
    return float(reciprocal_mass / external["weight"].sum())


def network_level_metrics(edges, month, network_scope, network_label, common_draws):
    working = edges.loc[edges["weight"] > 0].copy()
    if working.empty:
        raise ValueError(f"{month} {network_scope} {network_label} has no positive edges.")
    total_flow = float(working["weight"].sum())
    external = working.loc[working["origin"] != working["destination"]].copy()
    local_flow = float(working.loc[working["origin"] == working["destination"], "weight"].sum())

    edge_stats = entropy_metrics(working["weight"])
    origin_strength = working.groupby("origin")["weight"].sum()
    destination_strength = working.groupby("destination")["weight"].sum()
    origin_stats = entropy_metrics(origin_strength)
    destination_stats = entropy_metrics(destination_strength)

    external_flow = float(external["weight"].sum())
    distance_valid = external.loc[external["distance_km"].notna()].copy()
    covered_flow = float(distance_valid["weight"].sum())
    coverage = covered_flow / external_flow if external_flow > 0 else np.nan
    if distance_valid.empty:
        mean_distance = median_distance = p90_distance = np.nan
    else:
        mean_distance = float(np.average(distance_valid["distance_km"], weights=distance_valid["weight"]))
        median_distance = weighted_quantile(distance_valid["distance_km"], distance_valid["weight"], 0.50)
        p90_distance = weighted_quantile(distance_valid["distance_km"], distance_valid["weight"], 0.90)

    nodes = set(working["origin"]) | set(working["destination"])
    probabilities = working["weight"].to_numpy(dtype=float) / total_flow
    return {
        "month": month,
        "network_scope": network_scope,
        "network_label": network_label,
        "total_flow": total_flow,
        "positive_edge_count": int(len(working)),
        "active_node_count": int(len(nodes)),
        "active_origin_count": int(working["origin"].nunique()),
        "active_destination_count": int(working["destination"].nunique()),
        "local_flow_share": local_flow / total_flow,
        "external_flow_share": 1.0 - local_flow / total_flow,
        "weighted_reciprocity": weighted_reciprocity(working),
        "mean_external_distance_km": mean_distance,
        "median_external_distance_km": median_distance,
        "p90_external_distance_km": p90_distance,
        "distance_coverage_share": coverage,
        "edge_entropy": edge_stats["entropy"],
        "effective_edge_count": edge_stats["effective_count"],
        "normalized_edge_entropy": edge_stats["normalized_entropy"],
        "edge_hhi": edge_stats["hhi"],
        "largest_edge_share": edge_stats["largest_share"],
        "effective_origin_count": origin_stats["effective_count"],
        "origin_strength_hhi": origin_stats["hhi"],
        "effective_destination_count": destination_stats["effective_count"],
        "destination_strength_hhi": destination_stats["hhi"],
        "common_rarefaction_draws": int(common_draws),
        "rarefied_expected_edge_count": expected_occupied_count(probabilities, common_draws),
        "distance_method": DISTANCE_METHOD,
        "distance_crs": DISTANCE_CRS,
    }


def edge_set(edges):
    return set(map(tuple, edges.loc[edges["weight"] > 0, ["origin", "destination"]].to_numpy()))


def top_edge_set(edges, n):
    if n <= 0:
        return set()
    return set(map(tuple, (
        edges.loc[edges["weight"] > 0]
        .nlargest(n, "weight")[["origin", "destination"]]
        .to_numpy()
    )))


def network_similarity_metrics(hybrid_edges, onsite_edges, month, specification):
    hybrid = hybrid_edges[["origin", "destination", "weight"]].rename(columns={"weight": "hybrid_weight"})
    onsite = onsite_edges[["origin", "destination", "weight"]].rename(columns={"weight": "onsite_weight"})
    merged = hybrid.merge(onsite, on=["origin", "destination"], how="outer").fillna(0.0)

    h_set = set(map(tuple, hybrid[["origin", "destination"]].to_numpy()))
    o_set = set(map(tuple, onsite[["origin", "destination"]].to_numpy()))
    union = h_set | o_set
    intersection = h_set & o_set
    binary_jaccard = len(intersection) / len(union) if union else np.nan

    matched_n = min(len(h_set), len(o_set))
    h_top = top_edge_set(hybrid_edges, matched_n)
    o_top = top_edge_set(onsite_edges, matched_n)
    top_union = h_top | o_top
    density_matched_jaccard = len(h_top & o_top) / len(top_union) if top_union else np.nan

    h = merged["hybrid_weight"].to_numpy(dtype=float)
    o = merged["onsite_weight"].to_numpy(dtype=float)
    h_total, o_total = h.sum(), o.sum()
    h_norm = h / h_total if h_total > 0 else np.zeros_like(h)
    o_norm = o / o_total if o_total > 0 else np.zeros_like(o)

    denominator = np.linalg.norm(h) * np.linalg.norm(o)
    cosine = float(np.dot(h, o) / denominator) if denominator > 0 else np.nan
    weighted_jaccard_norm = (
        float(np.minimum(h_norm, o_norm).sum() / np.maximum(h_norm, o_norm).sum())
        if np.maximum(h_norm, o_norm).sum() > 0 else np.nan
    )
    shared_min_mass = float(np.minimum(h_norm, o_norm).sum())
    total_variation = float(0.5 * np.abs(h_norm - o_norm).sum())
    js_distance = float(jensenshannon(h_norm, o_norm, base=2.0)) if h_total > 0 and o_total > 0 else np.nan

    shared_mask = (merged["hybrid_weight"] > 0) & (merged["onsite_weight"] > 0)
    hybrid_shared_weight_share = (
        merged.loc[shared_mask, "hybrid_weight"].sum() / h_total if h_total > 0 else np.nan
    )
    onsite_shared_weight_share = (
        merged.loc[shared_mask, "onsite_weight"].sum() / o_total if o_total > 0 else np.nan
    )
    return {
        "month": month,
        "specification": specification,
        "binary_jaccard": binary_jaccard,
        "density_matched_jaccard": density_matched_jaccard,
        "cosine_similarity": cosine,
        "normalized_weighted_jaccard": weighted_jaccard_norm,
        "shared_minimum_probability_mass": shared_min_mass,
        "total_variation_reallocation_share": total_variation,
        "jensen_shannon_distance": js_distance,
        "hybrid_edge_count": len(h_set),
        "onsite_edge_count": len(o_set),
        "shared_edge_count": len(intersection),
        "hybrid_shared_weight_share": float(hybrid_shared_weight_share),
        "onsite_shared_weight_share": float(onsite_shared_weight_share),
    }


def external_partner_pairs(edges, role):
    if role == "residential":
        node_column, partner_column = "origin", "destination"
    elif role == "employment":
        node_column, partner_column = "destination", "origin"
    else:
        raise ValueError("role must be residential or employment")
    pairs = (
        edges.loc[
            (edges["weight"] > 0) & (edges["origin"] != edges["destination"]),
            [node_column, partner_column, "weight", "distance_km"],
        ]
        .groupby([node_column, partner_column], as_index=False)
        .agg(weight=("weight", "sum"), distance_km=("distance_km", "first"))
        .rename(columns={node_column: "district_id", partner_column: "partner_id"})
    )
    if not pairs.empty:
        pairs["external_total"] = pairs.groupby("district_id")["weight"].transform("sum")
        pairs["partner_probability"] = pairs["weight"] / pairs["external_total"]
    return pairs


def node_role_metrics(edges, role):
    if role == "residential":
        node_column, partner_column = "origin", "destination"
    elif role == "employment":
        node_column, partner_column = "destination", "origin"
    else:
        raise ValueError("role must be residential or employment")

    all_pairs = (
        edges.loc[edges["weight"] > 0, [node_column, partner_column, "weight", "distance_km"]]
        .groupby([node_column, partner_column], as_index=False)
        .agg(weight=("weight", "sum"), distance_km=("distance_km", "first"))
    )
    totals = (
        all_pairs.groupby(node_column, as_index=False)
        .agg(total_flow=("weight", "sum"), partner_count_raw=(partner_column, "nunique"))
        .rename(columns={node_column: "district_id"})
    )
    network_total = totals["total_flow"].sum()
    totals["network_flow_share"] = totals["total_flow"] / network_total if network_total > 0 else np.nan

    local = (
        all_pairs.loc[all_pairs[node_column] == all_pairs[partner_column]]
        .groupby(node_column)["weight"].sum().rename("local_flow")
    )
    external = all_pairs.loc[all_pairs[node_column] != all_pairs[partner_column]].copy()

    if external.empty:
        external_summary = pd.DataFrame(columns=[
            "district_id", "external_flow", "external_partner_count", "partner_entropy",
            "effective_partner_diversity", "normalized_partner_entropy",
            "largest_partner_share", "mean_external_distance_km", "distance_coverage_share",
        ])
    else:
        external["external_total"] = external.groupby(node_column)["weight"].transform("sum")
        external["partner_probability"] = external["weight"] / external["external_total"]
        external["entropy_component"] = -external["partner_probability"] * np.log(external["partner_probability"])
        summary = (
            external.groupby(node_column, as_index=False)
            .agg(
                external_flow=("weight", "sum"),
                external_partner_count=(partner_column, "nunique"),
                partner_entropy=("entropy_component", "sum"),
                largest_partner_share=("partner_probability", "max"),
            )
            .rename(columns={node_column: "district_id"})
        )
        summary["effective_partner_diversity"] = np.exp(summary["partner_entropy"])
        summary["normalized_partner_entropy"] = 0.0
        more_than_one = summary["external_partner_count"] > 1
        summary.loc[more_than_one, "normalized_partner_entropy"] = (
            summary.loc[more_than_one, "partner_entropy"]
            / np.log(summary.loc[more_than_one, "external_partner_count"])
        )

        distance_valid = external.loc[external["distance_km"].notna()].copy()
        if distance_valid.empty:
            distance_summary = pd.DataFrame(columns=[
                "district_id", "distance_covered_external_flow", "mean_external_distance_km"
            ])
        else:
            distance_valid["weighted_distance"] = distance_valid["weight"] * distance_valid["distance_km"]
            distance_summary = (
                distance_valid.groupby(node_column, as_index=False)
                .agg(
                    distance_covered_external_flow=("weight", "sum"),
                    weighted_distance=("weighted_distance", "sum"),
                )
                .rename(columns={node_column: "district_id"})
            )
            distance_summary["mean_external_distance_km"] = (
                distance_summary["weighted_distance"]
                / distance_summary["distance_covered_external_flow"]
            )
            distance_summary = distance_summary.drop(columns="weighted_distance")

        external_summary = summary.merge(distance_summary, on="district_id", how="left")
        external_summary["distance_covered_external_flow"] = pd.to_numeric(
            external_summary.get("distance_covered_external_flow"), errors="coerce"
        ).fillna(0.0)
        external_summary["distance_coverage_share"] = np.where(
            external_summary["external_flow"] > 0,
            external_summary["distance_covered_external_flow"] / external_summary["external_flow"],
            np.nan,
        )
        external_summary.loc[
            external_summary["distance_coverage_share"] < MIN_DISTANCE_COVERAGE_SHARE,
            "mean_external_distance_km",
        ] = np.nan

    result = totals.merge(local, left_on="district_id", right_index=True, how="left")
    result = result.merge(external_summary, on="district_id", how="left")
    result["local_flow"] = result["local_flow"].fillna(0.0)
    result["external_flow"] = pd.to_numeric(result.get("external_flow"), errors="coerce").fillna(0.0)
    result["external_partner_count"] = pd.to_numeric(
        result.get("external_partner_count"), errors="coerce"
    ).fillna(0).astype(int)
    result["external_flow_share"] = np.where(
        result["total_flow"] > 0, result["external_flow"] / result["total_flow"], np.nan
    )
    result["local_flow_share"] = np.where(
        result["total_flow"] > 0, result["local_flow"] / result["total_flow"], np.nan
    )
    result["role"] = role
    return result


def add_rarefied_partner_count(metrics, partner_pairs, prefix, common_draws):
    if partner_pairs.empty:
        metrics[f"{prefix}_rarefied_expected_partner_count"] = np.nan
        return metrics
    working = partner_pairs.merge(
        common_draws[["district_id", "common_partner_draws"]],
        on="district_id",
        how="inner",
        validate="many_to_one",
    )
    working = working.loc[working["common_partner_draws"] > 0].copy()
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        working["occupied_probability"] = -np.expm1(
            working["common_partner_draws"] * np.log1p(-working["partner_probability"])
        )
    expected = (
        working.groupby("district_id")["occupied_probability"]
        .sum().rename(f"{prefix}_rarefied_expected_partner_count")
    )
    return metrics.merge(expected, left_on="district_id", right_index=True, how="left")


def paired_node_premiums(hybrid_edges, onsite_edges, month, specification, role):
    hybrid_base = node_role_metrics(hybrid_edges, role)
    hybrid_metrics = hybrid_base.rename(
        columns={column: f"hybrid_{column}" for column in hybrid_base.columns if column not in ["district_id", "role"]}
    ).drop(columns="role")
    onsite_base = node_role_metrics(onsite_edges, role)
    onsite_metrics = onsite_base.rename(
        columns={column: f"onsite_{column}" for column in onsite_base.columns if column not in ["district_id", "role"]}
    ).drop(columns="role")

    paired = hybrid_metrics.merge(onsite_metrics, on="district_id", how="inner", validate="one_to_one")
    paired["common_partner_draws"] = np.floor(np.minimum(
        paired["hybrid_external_flow"], paired["onsite_external_flow"]
    ).clip(upper=NODE_RAREFACTION_DRAW_CAP)).astype(int)

    draw_lookup = paired[["district_id", "common_partner_draws"]]
    h_pairs = external_partner_pairs(hybrid_edges, role)
    o_pairs = external_partner_pairs(onsite_edges, role)
    paired = add_rarefied_partner_count(paired, h_pairs, "hybrid", draw_lookup)
    paired = add_rarefied_partner_count(paired, o_pairs, "onsite", draw_lookup)

    paired["common_support"] = (
        (paired["hybrid_external_flow"] > MIN_EXTERNAL_FLOW)
        & (paired["onsite_external_flow"] > MIN_EXTERNAL_FLOW)
        & (paired["hybrid_external_partner_count"] >= MIN_EXTERNAL_PARTNERS)
        & (paired["onsite_external_partner_count"] >= MIN_EXTERNAL_PARTNERS)
        & (paired["common_partner_draws"] > 0)
    )

    positive_effective = (
        (paired["hybrid_effective_partner_diversity"] > 0)
        & (paired["onsite_effective_partner_diversity"] > 0)
    )
    paired["log_effective_partner_premium"] = np.nan
    paired.loc[positive_effective, "log_effective_partner_premium"] = (
        np.log(paired.loc[positive_effective, "hybrid_effective_partner_diversity"])
        - np.log(paired.loc[positive_effective, "onsite_effective_partner_diversity"])
    )

    positive_rarefied = (
        (paired["hybrid_rarefied_expected_partner_count"] > 0)
        & (paired["onsite_rarefied_expected_partner_count"] > 0)
    )
    paired["log_rarefied_partner_premium"] = np.nan
    paired.loc[positive_rarefied, "log_rarefied_partner_premium"] = (
        np.log(paired.loc[positive_rarefied, "hybrid_rarefied_expected_partner_count"])
        - np.log(paired.loc[positive_rarefied, "onsite_rarefied_expected_partner_count"])
    )

    positive_distance = (
        (paired["hybrid_mean_external_distance_km"] > 0)
        & (paired["onsite_mean_external_distance_km"] > 0)
    )
    paired["log_mean_distance_premium"] = np.nan
    paired.loc[positive_distance, "log_mean_distance_premium"] = (
        np.log(paired.loc[positive_distance, "hybrid_mean_external_distance_km"])
        - np.log(paired.loc[positive_distance, "onsite_mean_external_distance_km"])
    )

    paired["effective_partner_premium_pct"] = 100.0 * np.expm1(paired["log_effective_partner_premium"])
    paired["rarefied_partner_premium_pct"] = 100.0 * np.expm1(paired["log_rarefied_partner_premium"])
    paired["distance_premium_pct"] = 100.0 * np.expm1(paired["log_mean_distance_premium"])
    paired["external_flow_share_difference"] = (
        paired["hybrid_external_flow_share"] - paired["onsite_external_flow_share"]
    )
    paired["largest_partner_share_difference"] = (
        paired["hybrid_largest_partner_share"] - paired["onsite_largest_partner_share"]
    )
    paired["month"] = month
    paired["specification"] = specification
    paired["role"] = role
    return paired
def district_role_hybrid_intensity(month_edges, role, external_only=True):
    """Flow-weighted hybrid-work intensity for each district and role."""
    if role == "residential":
        node_column = "origin"
    elif role == "employment":
        node_column = "destination"
    else:
        raise ValueError("role must be residential or employment")

    working = month_edges.loc[month_edges["layer_weight"] > 0].copy()
    if external_only:
        working = working.loc[working["origin"] != working["destination"]].copy()
    working["hybrid_intensity_component"] = (
        working["recurrence_band"].map(RECURRENCE_INTENSITY)
        * working["layer_weight"]
    )
    result = (
        working.groupby(node_column, as_index=False)
        .agg(
            intensity_flow=("layer_weight", "sum"),
            intensity_weighted_flow=("hybrid_intensity_component", "sum"),
        )
        .rename(columns={node_column: "district_id"})
    )
    result["hybrid_intensity"] = np.where(
        result["intensity_flow"] > 0,
        result["intensity_weighted_flow"] / result["intensity_flow"],
        np.nan,
    )
    result["role"] = role
    return result


def generic_network_similarity(edges_a, edges_b, month, band_a, band_b):
    """Pairwise similarity for any two recurrence layers."""
    base = network_similarity_metrics(
        edges_a,
        edges_b,
        month=month,
        specification=f"{band_a}__{band_b}",
    )
    rename = {
        "hybrid_edge_count": "band_a_edge_count",
        "onsite_edge_count": "band_b_edge_count",
        "hybrid_shared_weight_share": "band_a_shared_weight_share",
        "onsite_shared_weight_share": "band_b_shared_weight_share",
    }
    base = {rename.get(key, key): value for key, value in base.items()}
    base.update({
        "band_a": band_a,
        "band_b": band_b,
        "band_a_intensity": RECURRENCE_INTENSITY[band_a],
        "band_b_intensity": RECURRENCE_INTENSITY[band_b],
        "intensity_gap": abs(
            RECURRENCE_INTENSITY[band_a] - RECURRENCE_INTENSITY[band_b]
        ),
        "layer_step_gap": abs(
            RECURRENCE_BAND_ORDER.index(band_a)
            - RECURRENCE_BAND_ORDER.index(band_b)
        ),
    })
    return base


def add_fixed_partner_coverage(metrics, partner_pairs, draws, output_column):
    """Expected number of occupied external partners at one common draw."""
    result = metrics.copy()
    result[output_column] = np.nan
    if partner_pairs.empty or draws <= 0:
        return result
    working = partner_pairs.copy()
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        working["occupied_probability"] = -np.expm1(
            draws * np.log1p(-working["partner_probability"])
        )
    expected = (
        working.groupby(["month", "role", "district_id"])["occupied_probability"]
        .sum()
        .rename(output_column)
        .reset_index()
    )
    result = result.drop(columns=[output_column]).merge(
        expected,
        on=["month", "role", "district_id"],
        how="left",
        validate="one_to_one",
    )
    valid_support = (
        (result["external_flow"] >= draws)
        & (result["external_partner_count"] >= MIN_EXTERNAL_PARTNERS)
    )
    result.loc[~valid_support, output_column] = np.nan
    return result


## 8. Period-level spectral hierarchy and formal regression helpers

In [ ]:
def spectral_centrality(edges):
    positive = edges.loc[
        edges["weight"] > 0, ["origin", "destination", "weight"]
    ].copy()
    nodes = sorted(set(positive["origin"]) | set(positive["destination"]))
    if not nodes:
        return pd.DataFrame(columns=["district_id", "pagerank", "hub_score", "authority_score"])

    node_index = {node: index for index, node in enumerate(nodes)}
    rows = positive["origin"].map(node_index).to_numpy()
    columns = positive["destination"].map(node_index).to_numpy()
    weights = positive["weight"].to_numpy(dtype=float)
    weights = weights / weights.sum()
    matrix = sparse.csr_matrix((weights, (rows, columns)), shape=(len(nodes), len(nodes)))

    out_strength = np.asarray(matrix.sum(axis=1)).ravel()
    inverse_out = np.zeros_like(out_strength)
    valid_out = out_strength > 0
    inverse_out[valid_out] = 1.0 / out_strength[valid_out]
    transition = sparse.diags(inverse_out) @ matrix

    alpha = 0.85
    pagerank = np.full(len(nodes), 1.0 / len(nodes))
    dangling = ~valid_out
    for _ in range(500):
        previous = pagerank.copy()
        dangling_mass = previous[dangling].sum()
        pagerank = alpha * (
            transition.T @ previous + dangling_mass / len(nodes)
        ) + (1.0 - alpha) / len(nodes)
        pagerank = np.asarray(pagerank).ravel()
        if np.abs(pagerank - previous).sum() < 1e-12:
            break

    hub = np.full(len(nodes), 1.0 / np.sqrt(len(nodes)))
    authority = hub.copy()
    for _ in range(500):
        previous_hub = hub.copy()
        previous_authority = authority.copy()
        authority = matrix.T @ hub
        authority_norm = np.linalg.norm(authority)
        if authority_norm > 0:
            authority = authority / authority_norm
        hub = matrix @ authority
        hub_norm = np.linalg.norm(hub)
        if hub_norm > 0:
            hub = hub / hub_norm
        if max(
            np.max(np.abs(hub - previous_hub)),
            np.max(np.abs(authority - previous_authority)),
        ) < 1e-12:
            break

    return pd.DataFrame({
        "district_id": nodes,
        "pagerank": pagerank,
        "hub_score": np.asarray(hub).ravel(),
        "authority_score": np.asarray(authority).ravel(),
    })


def coefficient_table(fitted, metadata):
    confidence = fitted.conf_int()
    rows = []
    for term in fitted.params.index:
        row = dict(metadata)
        row.update({
            "term": term,
            "coefficient": float(fitted.params[term]),
            "std_error": float(fitted.bse[term]),
            "test_statistic": float(fitted.tvalues[term]),
            "p_value": float(fitted.pvalues[term]),
            "ci_low": float(confidence.loc[term, 0]),
            "ci_high": float(confidence.loc[term, 1]),
            "nobs": int(fitted.nobs),
            "r_squared": float(getattr(fitted, "rsquared", np.nan)),
            "adjusted_r_squared": float(getattr(fitted, "rsquared_adj", np.nan)),
        })
        rows.append(row)
    return rows


def paired_month_test(data, metric, specification):
    part = data.loc[data["specification"] == specification, ["month", "network_label", metric]].dropna()
    wide = part.pivot(index="month", columns="network_label", values=metric).dropna()
    if not {"Hybrid", "Onsite"}.issubset(wide.columns):
        return None
    difference = wide["Hybrid"] - wide["Onsite"]
    t_result = stats.ttest_rel(wide["Hybrid"], wide["Onsite"], nan_policy="omit")
    try:
        w_result = stats.wilcoxon(difference)
        w_stat, w_p = float(w_result.statistic), float(w_result.pvalue)
    except ValueError:
        w_stat, w_p = np.nan, np.nan
    return {
        "specification": specification,
        "metric": metric,
        "months": int(len(wide)),
        "hybrid_mean": float(wide["Hybrid"].mean()),
        "onsite_mean": float(wide["Onsite"].mean()),
        "mean_difference": float(difference.mean()),
        "median_difference": float(difference.median()),
        "share_positive_difference": float((difference > 0).mean()),
        "paired_t_statistic": float(t_result.statistic),
        "paired_t_p_value": float(t_result.pvalue),
        "wilcoxon_statistic": w_stat,
        "wilcoxon_p_value": w_p,
    }


def average_prediction_curve(fitted, reference_data, x_column, moderator_column, moderator_values, grid):
    design_info = fitted.model.data.design_info
    covariance = np.asarray(fitted.cov_params())
    parameters = np.asarray(fitted.params)
    rows = []
    for moderator_value in moderator_values:
        for x_value in grid:
            new_data = reference_data.copy()
            new_data[x_column] = x_value
            new_data[moderator_column] = moderator_value
            matrix = patsy.build_design_matrices([design_info], new_data, return_type="dataframe")[0]
            average_design = np.asarray(matrix.mean(axis=0), dtype=float)
            estimate = float(average_design @ parameters)
            variance = float(average_design @ covariance @ average_design)
            standard_error = math.sqrt(max(variance, 0.0))
            rows.append({
                x_column: x_value,
                moderator_column: moderator_value,
                "prediction": estimate,
                "ci_low": estimate - 1.96 * standard_error,
                "ci_high": estimate + 1.96 * standard_error,
            })
    return pd.DataFrame(rows)
def iterative_two_way_demean(frame, columns, entity_column, time_column):
    """Alternating-projection demeaning for entity and time fixed effects."""
    transformed = frame[columns].astype(float).copy()
    for iteration in range(FE_DEMEAN_MAX_ITER):
        previous = transformed.to_numpy(copy=True)
        transformed = transformed - transformed.groupby(frame[entity_column]).transform("mean")
        transformed = transformed - transformed.groupby(frame[time_column]).transform("mean")
        difference = np.nanmax(np.abs(transformed.to_numpy() - previous))
        if np.isfinite(difference) and difference < FE_DEMEAN_TOLERANCE:
            return transformed, iteration + 1
    warnings.warn("Two-way demeaning reached the iteration limit.")
    return transformed, FE_DEMEAN_MAX_ITER


def fit_two_way_fixed_effect_model(frame, outcome, predictors):
    required = [outcome, *predictors, "district_id", "month"]
    data = frame[required].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if data.empty:
        raise ValueError(f"No valid observations for {outcome}.")
    transformed, iterations = iterative_two_way_demean(
        data,
        [outcome, *predictors],
        entity_column="district_id",
        time_column="month",
    )
    y = transformed[outcome]
    X = transformed[predictors]
    keep = y.notna() & X.notna().all(axis=1)
    y = y.loc[keep]
    X = X.loc[keep]
    groups = data.loc[keep, "district_id"]
    fitted = sm.OLS(y, X).fit(
        cov_type="cluster",
        cov_kwds={"groups": groups, "use_correction": True},
    )
    fitted._demeaning_iterations = iterations
    fitted._analysis_data = data.loc[keep].copy()
    return fitted


def average_adjusted_curve(fitted, reference_data, x_column, x_values, fixed_values):
    design_info = fitted.model.data.design_info
    params = fitted.params.to_numpy(dtype=float)
    covariance = fitted.cov_params().to_numpy(dtype=float)
    rows = []
    for x_value in x_values:
        scenario = reference_data.copy()
        scenario[x_column] = x_value
        for column, value in fixed_values.items():
            scenario[column] = value
        matrix = patsy.build_design_matrices(
            [design_info], scenario, return_type="dataframe"
        )[0]
        mean_design = matrix.mean(axis=0).to_numpy(dtype=float)
        prediction = float(mean_design @ params)
        variance = float(mean_design @ covariance @ mean_design)
        standard_error = math.sqrt(max(variance, 0.0))
        rows.append({
            x_column: x_value,
            "prediction_log": prediction,
            "standard_error_log": standard_error,
            "ci_low_log": prediction - 1.96 * standard_error,
            "ci_high_log": prediction + 1.96 * standard_error,
            "prediction": float(np.exp(prediction)),
            "ci_low": float(np.exp(prediction - 1.96 * standard_error)),
            "ci_high": float(np.exp(prediction + 1.96 * standard_error)),
        })
    return pd.DataFrame(rows)


def intensity_marginal_effects(
    fitted,
    frame,
    intensity_column=PRIMARY_INTENSITY_COLUMN,
):
    """District-specific intensity slopes from the context interaction model."""
    b = fitted.params
    v = fitted.cov_params()

    centered = f"{intensity_column}_c"
    main = centered
    urban = f"{centered}:settlement_urbanity_c"
    position = f"{centered}:onsite_network_position_c"

    if main not in b.index:
        raise KeyError(
            f"The fitted model does not contain the required intensity term: {main}"
        )

    result = frame.copy()
    u = result["settlement_urbanity_c"].to_numpy(dtype=float)
    p = result["onsite_network_position_c"].to_numpy(dtype=float)

    slope = (
        b.get(main, 0.0)
        + b.get(urban, 0.0) * u
        + b.get(position, 0.0) * p
    )

    var = np.full(len(result), float(v.loc[main, main]))

    if urban in b.index:
        var += (u ** 2) * float(v.loc[urban, urban])
        var += 2 * u * float(v.loc[main, urban])

    if position in b.index:
        var += (p ** 2) * float(v.loc[position, position])
        var += 2 * p * float(v.loc[main, position])

    if urban in b.index and position in b.index:
        var += 2 * u * p * float(v.loc[urban, position])

    se = np.sqrt(np.maximum(var, 0.0))

    result["intensity_slope_log"] = slope
    result["intensity_slope_se"] = se
    result["intensity_slope_ci_low"] = slope - 1.96 * se
    result["intensity_slope_ci_high"] = slope + 1.96 * se

    for source, target in [
        ("intensity_slope_log", "intensity_effect_per_0_1_pct"),
        ("intensity_slope_ci_low", "intensity_effect_per_0_1_ci_low_pct"),
        ("intensity_slope_ci_high", "intensity_effect_per_0_1_ci_high_pct"),
    ]:
        result[target] = 100.0 * np.expm1(
            INTENSITY_EFFECT_INCREMENT * result[source]
        )

    return result



def coefficient_table(fitted, metadata=None, **metadata_kwargs):
    """Return a tidy DataFrame for formula and demeaned OLS models."""
    meta = dict(metadata or {})
    meta.update(metadata_kwargs)
    confidence = fitted.conf_int()
    rows = []
    for term in fitted.params.index:
        row = dict(meta)
        row.update({
            "term": term,
            "estimate": float(fitted.params[term]),
            "robust_se": float(fitted.bse[term]),
            "test_statistic": float(fitted.tvalues[term]),
            "p_value": float(fitted.pvalues[term]),
            "ci_low": float(confidence.loc[term, 0]),
            "ci_high": float(confidence.loc[term, 1]),
            "n": int(fitted.nobs),
            "r_squared": float(getattr(fitted, "rsquared", np.nan)),
            "adjusted_r_squared": float(getattr(fitted, "rsquared_adj", np.nan)),
        })
        rows.append(row)
    return pd.DataFrame(rows)


## 9. Monthly processing across all six hybrid-work intensity layers


In [ ]:
def cache_paths(month):
    safe = month.replace("-", "_")
    return {
        "layer_metrics": CACHE_DIR / f"layer_metrics_{safe}.parquet",
        "layer_edges": CACHE_DIR / f"layer_edges_{safe}.parquet",
        "pairwise_similarity": CACHE_DIR / f"pairwise_layer_similarity_{safe}.parquet",
        "district_metrics_base": CACHE_DIR / f"district_metrics_base_{safe}.parquet",
        "district_partner_pairs": CACHE_DIR / f"district_partner_pairs_{safe}.parquet",
    }


def cache_is_valid(paths):
    if not all(path.exists() for path in paths.values()):
        return False
    try:
        for path in paths.values():
            signature = pd.read_parquet(path, columns=["analysis_signature"])["analysis_signature"]
            if signature.empty or not signature.eq(ANALYSIS_SIGNATURE).all():
                return False
        return True
    except Exception:
        return False


processing_rows = []
for month in VALID_MONTHS:
    paths = cache_paths(month)
    if (
        REUSE_MONTHLY_CACHE
        and month not in FORCE_REPROCESS_MONTHS
        and cache_is_valid(paths)
    ):
        print(f"[{month}] Reusing valid intensity-gradient cache.")
        processing_rows.append({"month": month, "status": "reused"})
        continue

    print(f"[{month}] Reading Stage-0 edges and processing all six intensity layers.")
    month_edges = read_stage0_month(VALID_FILE_BY_MONTH[month], month)
    distance_qc = distance_coverage_summary(
        month_edges.rename(columns={"layer_weight": "weight"}),
        weight_column="weight",
    )
    if (
        not np.isfinite(distance_qc["distance_coverage_share"])
        or distance_qc["distance_coverage_share"] < MIN_DISTANCE_COVERAGE_SHARE
    ):
        raise RuntimeError(
            f"{month} distance coverage is insufficient: "
            f"{distance_qc['distance_coverage_share']:.2%}."
        )

    layer_totals = (
        month_edges.groupby("recurrence_band")["layer_weight"]
        .sum().reindex(RECURRENCE_BAND_ORDER)
    )
    if layer_totals.isna().any():
        raise RuntimeError(f"{month} lacks one or more recurrence layers.")
    layer_common_draws = int(max(1, min(
        NETWORK_RAREFACTION_DRAW_CAP,
        math.floor(layer_totals.min()),
    )))

    layer_metric_rows = []
    layer_edge_parts = []
    layer_networks = {}
    for band in RECURRENCE_BAND_ORDER:
        layer = network_from_bands(month_edges, [band])
        layer_networks[band] = layer
        row = network_level_metrics(
            layer,
            month,
            network_scope="recurrence_layer",
            network_label=band,
            common_draws=layer_common_draws,
        )
        row.update({
            "recurrence_band": band,
            "recurrence_midpoint": RECURRENCE_MIDPOINT[band],
            "hybrid_work_intensity": RECURRENCE_INTENSITY[band],
        })
        layer_metric_rows.append(row)

        part = layer.copy()
        part["month"] = month
        part["recurrence_band"] = band
        part["hybrid_work_intensity"] = RECURRENCE_INTENSITY[band]
        layer_edge_parts.append(part)

    pairwise_rows = []
    for band_a, band_b in combinations(RECURRENCE_BAND_ORDER, 2):
        pairwise_rows.append(generic_network_similarity(
            layer_networks[band_a], layer_networks[band_b], month, band_a, band_b
        ))

    full_network = network_from_bands(month_edges, RECURRENCE_BAND_ORDER)
    district_parts = []
    partner_parts = []
    for role in ["residential", "employment"]:
        metrics = node_role_metrics(full_network, role)
        metrics["month"] = month
        external_intensity = district_role_hybrid_intensity(
            month_edges, role, external_only=True
        ).rename(columns={
            "hybrid_intensity": "hybrid_intensity_external",
            "intensity_flow": "intensity_external_flow",
            "intensity_weighted_flow": "intensity_external_weighted_flow",
        }).drop(columns="role")
        all_intensity = district_role_hybrid_intensity(
            month_edges, role, external_only=False
        ).rename(columns={
            "hybrid_intensity": "hybrid_intensity_all_flows",
            "intensity_flow": "intensity_all_flow",
            "intensity_weighted_flow": "intensity_all_weighted_flow",
        }).drop(columns="role")
        metrics = metrics.merge(
            external_intensity, on="district_id", how="left", validate="one_to_one"
        ).merge(
            all_intensity, on="district_id", how="left", validate="one_to_one"
        )
        district_parts.append(metrics)

        pairs = external_partner_pairs(full_network, role)
        pairs["month"] = month
        pairs["role"] = role
        partner_parts.append(pairs)

    outputs = {
        "layer_metrics": pd.DataFrame(layer_metric_rows),
        "layer_edges": pd.concat(layer_edge_parts, ignore_index=True),
        "pairwise_similarity": pd.DataFrame(pairwise_rows),
        "district_metrics_base": pd.concat(district_parts, ignore_index=True),
        "district_partner_pairs": pd.concat(partner_parts, ignore_index=True),
    }
    for key, frame in outputs.items():
        frame["analysis_signature"] = ANALYSIS_SIGNATURE
        atomic_to_parquet(frame, paths[key])

    processing_rows.append({
        "month": month,
        "status": "processed",
        "distance_coverage_share": distance_qc["distance_coverage_share"],
        "layer_metric_rows": len(outputs["layer_metrics"]),
        "pairwise_similarity_rows": len(outputs["pairwise_similarity"]),
        "district_metric_rows": len(outputs["district_metrics_base"]),
        "partner_pair_rows": len(outputs["district_partner_pairs"]),
    })

processing_log = pd.DataFrame(processing_rows)
atomic_to_csv(processing_log, LOG_DIR / "monthly_processing_log.csv")
display(processing_log)


## 10. Assemble monthly intensity-gradient outputs and define common partner coverage


In [ ]:
def read_cache_series(key):
    parts = []
    for month in VALID_MONTHS:
        path = cache_paths(month)[key]
        frame = pd.read_parquet(path)
        if not frame["analysis_signature"].eq(ANALYSIS_SIGNATURE).all():
            raise RuntimeError(f"Cache signature mismatch: {path}")
        parts.append(frame.drop(columns="analysis_signature"))
    return pd.concat(parts, ignore_index=True)


layer_metrics = read_cache_series("layer_metrics")
monthly_layer_edges = read_cache_series("layer_edges")
monthly_pairwise_similarity = read_cache_series("pairwise_similarity")
district_metrics_base = read_cache_series("district_metrics_base")
district_partner_pairs = read_cache_series("district_partner_pairs")


# Recalculate network connections at one common flow scale across every
# month and intensity layer.
global_layer_common_draws = int(max(1, min(
    NETWORK_RAREFACTION_DRAW_CAP,
    math.floor(float(layer_metrics["total_flow"].min())),
)))

connection_rows = []
for (month, band), frame in monthly_layer_edges.groupby(
    ["month", "recurrence_band"], sort=False
):
    probabilities = frame["weight"].to_numpy(dtype=float)
    probabilities = probabilities / probabilities.sum()
    connection_rows.append({
        "month": month,
        "recurrence_band": band,
        "network_connections_common_scale": expected_occupied_count(
            probabilities,
            global_layer_common_draws,
        ),
    })

connection_counts = pd.DataFrame(connection_rows)
layer_metrics = layer_metrics.merge(
    connection_counts,
    on=["month", "recurrence_band"],
    how="left",
    validate="one_to_one",
)


# Standardised partner-coverage measures remain available as robustness
# checks. They no longer determine the main analytical sample.
eligible_draws = district_metrics_base.loc[
    (district_metrics_base["external_flow"] > MIN_EXTERNAL_FLOW)
    & (district_metrics_base["external_partner_count"] >= MIN_EXTERNAL_PARTNERS),
    "external_flow",
].dropna()

if eligible_draws.empty:
    raise RuntimeError(
        "No eligible district-month observations for standardised partner coverage."
    )

coverage_draws = {}
for quantile in [
    PARTNER_COVERAGE_DRAW_QUANTILE,
    *PARTNER_COVERAGE_SENSITIVITY_QUANTILES,
]:
    draws = int(max(1, min(
        NODE_RAREFACTION_DRAW_CAP,
        math.floor(float(eligible_draws.quantile(quantile))),
    )))
    coverage_draws[quantile] = draws

main_partner_draws = coverage_draws[PARTNER_COVERAGE_DRAW_QUANTILE]

district_month = district_metrics_base.copy()

for quantile, draws in sorted(coverage_draws.items()):
    suffix = str(int(round(quantile * 100))).zfill(2)
    district_month = add_fixed_partner_coverage(
        district_month,
        district_partner_pairs,
        draws,
        output_column=f"partner_coverage_rarefied_q{suffix}",
    )

# Main partner coverage: observed number of external district partners.
# External flow is controlled in every panel model.
district_month["partner_coverage"] = pd.to_numeric(
    district_month["external_partner_count"],
    errors="coerce",
).fillna(0.0)

# Preserve structural zeros. Distance is not assigned a zero because an
# average external distance is undefined when no external relation exists.
no_external = district_month["external_flow"].fillna(0).le(0)

if INCLUDE_STRUCTURAL_ZEROS_COVERAGE:
    district_month.loc[no_external, "partner_coverage"] = 0.0

if INCLUDE_STRUCTURAL_ZEROS_DIVERSITY:
    district_month.loc[
        no_external,
        "effective_partner_diversity",
    ] = 0.0
    district_month.loc[no_external, "partner_entropy"] = 0.0
    district_month.loc[no_external, "normalized_partner_entropy"] = 0.0
    district_month.loc[no_external, "largest_partner_share"] = 0.0

district_month.loc[
    no_external,
    "mean_external_distance_km",
] = DISTANCE_WITHOUT_EXTERNAL_RELATION


district_month["year"] = district_month["month"].str[:4].astype(int)
district_month["province_code"] = district_month["district_id"].map(
    province_code_from_district
)

district_month = district_month.merge(
    ghs_profile,
    on="district_id",
    how="left",
    validate="many_to_one",
)


district_month_ghs_audit = pd.DataFrame([{
    "district_month_rows": int(len(district_month)),
    "district_month_unique_districts": int(
        district_month["district_id"].nunique()
    ),
    "matched_rows": int(
        district_month["settlement_urbanity"].notna().sum()
    ),
    "row_match_share": float(
        district_month["settlement_urbanity"].notna().mean()
    ),
    "matched_unique_districts": int(
        district_month.loc[
            district_month["settlement_urbanity"].notna(),
            "district_id",
        ].nunique()
    ),
    "unique_district_match_share": float(
        district_month.groupby("district_id")["settlement_urbanity"]
        .first()
        .notna()
        .mean()
    ),
}])

atomic_to_csv(
    district_month_ghs_audit,
    LOG_DIR / "district_month_ghs_merge_audit.csv",
)
display(district_month_ghs_audit)

district_month_unique_match_share = float(
    district_month_ghs_audit.loc[0, "unique_district_match_share"]
)

if (
    REQUIRE_GHS_PROFILE
    and district_month_unique_match_share < MIN_GHS_MATCH_SHARE
):
    raise RuntimeError(
        "The district-month panel matched too few districts to the GHSL profile: "
        f"{district_month_unique_match_share:.2%} < "
        f"{MIN_GHS_MATCH_SHARE:.2%}."
    )


district_month["common_partner_draws"] = main_partner_draws
district_month["primary_intensity"] = district_month[
    PRIMARY_INTENSITY_COLUMN
]

# Independent validity flags. One outcome can no longer remove an
# otherwise valid observation from another outcome.
district_month["valid_intensity"] = (
    district_month[PRIMARY_INTENSITY_COLUMN].notna()
    & district_month["total_flow"].gt(0)
)
district_month["valid_partner_coverage"] = (
    district_month["valid_intensity"]
    & district_month["partner_coverage"].notna()
)
district_month["valid_partner_diversity"] = (
    district_month["valid_intensity"]
    & district_month["effective_partner_diversity"].notna()
)
district_month["valid_mean_external_distance"] = (
    district_month["valid_intensity"]
    & district_month["mean_external_distance_km"].gt(0)
)

# Transformations used by the maximum-available-sample models.
district_month["log1p_partner_coverage"] = np.log1p(
    district_month["partner_coverage"].clip(lower=0)
)
district_month["log1p_effective_partner_diversity"] = np.log1p(
    district_month["effective_partner_diversity"].clip(lower=0)
)
district_month["log_mean_external_distance_km"] = np.where(
    district_month["mean_external_distance_km"] > 0,
    np.log(district_month["mean_external_distance_km"]),
    np.nan,
)
district_month["log_external_flow"] = np.log1p(
    district_month["external_flow"].clip(lower=0)
)
district_month["log_total_flow"] = np.log1p(
    district_month["total_flow"].clip(lower=0)
)

coverage_draw_diagnostics = pd.DataFrame([
    {
        "quantile": quantile,
        "draws": draws,
        "eligible_observations": int(len(eligible_draws)),
        "share_eligible_external_flow_at_least_draw": float(
            (eligible_draws >= draws).mean()
        ),
    }
    for quantile, draws in sorted(coverage_draws.items())
])

outcome_sample_diagnostics = pd.DataFrame([
    {
        "role": role,
        "outcome": outcome,
        "valid_rows": int(
            district_month.loc[
                district_month["role"].eq(role),
                flag,
            ].sum()
        ),
        "unique_districts": int(
            district_month.loc[
                district_month["role"].eq(role)
                & district_month[flag],
                "district_id",
            ].nunique()
        ),
        "months": int(
            district_month.loc[
                district_month["role"].eq(role)
                & district_month[flag],
                "month",
            ].nunique()
        ),
    }
    for role in ["residential", "employment"]
    for outcome, flag in [
        ("hybrid_intensity", "valid_intensity"),
        ("partner_coverage", "valid_partner_coverage"),
        ("partner_diversity", "valid_partner_diversity"),
        ("mean_external_distance", "valid_mean_external_distance"),
    ]
])

atomic_to_parquet(
    layer_metrics,
    TABLE_MAIN_DIR / "monthly_layer_network_metrics.parquet",
)
atomic_to_parquet(
    monthly_pairwise_similarity,
    TABLE_MAIN_DIR / "monthly_pairwise_layer_similarity.parquet",
)
atomic_to_parquet(
    district_month,
    TABLE_MAIN_DIR / "district_month_role_intensity_metrics.parquet",
)
atomic_to_csv(
    coverage_draw_diagnostics,
    TABLE_MAIN_DIR / "partner_coverage_draw_diagnostics.csv",
)
atomic_to_csv(
    outcome_sample_diagnostics,
    TABLE_MAIN_DIR / "outcome_specific_sample_diagnostics.csv",
)

print("Layer metric rows:", len(layer_metrics))
print("Pairwise similarity rows:", len(monthly_pairwise_similarity))
print("District-month-role rows:", len(district_month))
print("Global network common-flow draw:", global_layer_common_draws)
print("Rarefied partner-coverage draw:", main_partner_draws)
print("Primary district intensity:", PRIMARY_INTENSITY_COLUMN)
print("Primary partner coverage:", PRIMARY_PARTNER_COVERAGE_DEFINITION)

display(coverage_draw_diagnostics)
display(outcome_sample_diagnostics)


# Part I. National job–home network structure across hybrid-work intensity

This section uses all six layers. It first compares the connections and flow weights of every pair of layers. It then examines whether national network properties follow a stable hybrid-work intensity gradient in the period aggregate and in each month.


## 11. Period-aggregated layer networks and pairwise similarity


In [ ]:
period_layer_edges = (
    monthly_layer_edges.groupby(
        ["recurrence_band", "origin", "destination"], as_index=False
    )
    .agg(weight=("weight", "sum"), distance_km=("distance_km", "first"))
)
period_layer_edges["hybrid_work_intensity"] = period_layer_edges["recurrence_band"].map(
    RECURRENCE_INTENSITY
)
period_layer_edges["layer_total_flow"] = period_layer_edges.groupby(
    "recurrence_band"
)["weight"].transform("sum")
period_layer_edges["weight_share"] = (
    period_layer_edges["weight"] / period_layer_edges["layer_total_flow"]
)

period_layer_networks = {
    band: period_layer_edges.loc[
        period_layer_edges["recurrence_band"] == band,
        ["origin", "destination", "weight", "distance_km"],
    ].copy()
    for band in RECURRENCE_BAND_ORDER
}
period_pairwise_rows = []
for band_a, band_b in combinations(RECURRENCE_BAND_ORDER, 2):
    period_pairwise_rows.append(generic_network_similarity(
        period_layer_networks[band_a],
        period_layer_networks[band_b],
        month="2022-2024",
        band_a=band_a,
        band_b=band_b,
    ))
period_pairwise_similarity = pd.DataFrame(period_pairwise_rows)

monthly_pairwise_summary = (
    monthly_pairwise_similarity.groupby(
        ["band_a", "band_b", "layer_step_gap", "intensity_gap"], as_index=False
    )
    .agg(
        mean_binary_jaccard=("binary_jaccard", "mean"),
        median_binary_jaccard=("binary_jaccard", "median"),
        mean_cosine_similarity=("cosine_similarity", "mean"),
        median_cosine_similarity=("cosine_similarity", "median"),
        mean_weighted_jaccard=("normalized_weighted_jaccard", "mean"),
        mean_reallocation_share=("total_variation_reallocation_share", "mean"),
        months=("month", "nunique"),
    )
)

atomic_to_parquet(
    period_layer_edges,
    MAP_INPUT_DIR / "period_layer_network_edges_map_ready.parquet",
)
atomic_to_csv(
    period_pairwise_similarity,
    TABLE_MAIN_DIR / "period_pairwise_layer_similarity.csv",
)
atomic_to_csv(
    monthly_pairwise_summary,
    TABLE_MAIN_DIR / "monthly_pairwise_layer_similarity_summary.csv",
)

display(period_pairwise_similarity.round(4))


## 12. Pairwise similarity heatmaps


In [ ]:
def similarity_matrix(frame, value_column):
    matrix = pd.DataFrame(
        np.eye(len(RECURRENCE_BAND_ORDER)),
        index=RECURRENCE_BAND_ORDER,
        columns=RECURRENCE_BAND_ORDER,
        dtype=float,
    )
    for row in frame.itertuples(index=False):
        matrix.loc[row.band_a, row.band_b] = getattr(row, value_column)
        matrix.loc[row.band_b, row.band_a] = getattr(row, value_column)
    return matrix

jaccard_matrix = similarity_matrix(period_pairwise_similarity, "binary_jaccard")
cosine_matrix = similarity_matrix(period_pairwise_similarity, "cosine_similarity")

figure, axes = plt.subplots(1, 2, figsize=(12.8, 5.5))
for axis, matrix, title, vmin in [
    (axes[0], jaccard_matrix, "Shared links (Jaccard)", 0.0),
    (axes[1], cosine_matrix, "Similarity of link weights (cosine)", 0.0),
]:
    image = axis.imshow(matrix.to_numpy(), cmap="viridis", vmin=vmin, vmax=1.0)
    axis.set_xticks(range(len(RECURRENCE_BAND_ORDER)), RECURRENCE_BAND_ORDER)
    axis.set_yticks(range(len(RECURRENCE_BAND_ORDER)), RECURRENCE_BAND_ORDER)
    axis.set_xlabel("Workplace recurrence (days in a 14-day period)")
    axis.set_ylabel("Workplace recurrence (days in a 14-day period)")
    axis.set_title(title)
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            value = matrix.iloc[i, j]
            colour = "white" if value < 0.55 else "black"
            axis.text(j, i, f"{value:.2f}", ha="center", va="center", color=colour, fontsize=8)
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
figure.suptitle("Pairwise similarity of job–home networks across hybrid-work intensity")
figure.subplots_adjust(top=0.84, wspace=0.30)
save_figure(
    figure,
    "figure_1_pairwise_similarity_across_intensity_layers",
    FIGURE_MAIN_DIR,
)


## 13. Period gradient in national network structure


In [ ]:
period_common_draws = int(max(1, min(
    NETWORK_RAREFACTION_DRAW_CAP,
    math.floor(min(
        network["weight"].sum()
        for network in period_layer_networks.values()
    )),
)))
period_metric_rows = []
for band in RECURRENCE_BAND_ORDER:
    row = network_level_metrics(
        period_layer_networks[band],
        month="2022-2024",
        network_scope="period_recurrence_layer",
        network_label=band,
        common_draws=period_common_draws,
    )
    row.update({
        "recurrence_band": band,
        "recurrence_midpoint": RECURRENCE_MIDPOINT[band],
        "hybrid_work_intensity": RECURRENCE_INTENSITY[band],
    })
    period_metric_rows.append(row)
period_layer_metrics = pd.DataFrame(period_metric_rows).sort_values(
    "hybrid_work_intensity", ascending=False
)

metrics_to_plot = [
    ("external_flow_share", "Inter-district flow share"),
    ("mean_external_distance_km", "Mean external distance (km)"),
    ("rarefied_expected_edge_count", "Network connections at a common flow scale"),
    ("edge_hhi", "Flow concentration (HHI)"),
]
figure, axes = plt.subplots(2, 2, figsize=(12.5, 9.0))
for axis, (column, label) in zip(axes.flat, metrics_to_plot):
    frame = period_layer_metrics.sort_values("hybrid_work_intensity")
    axis.plot(
        frame["hybrid_work_intensity"], frame[column], marker="o", linewidth=1.8
    )
    for row in frame.itertuples(index=False):
        axis.annotate(
            row.recurrence_band,
            (row.hybrid_work_intensity, getattr(row, column)),
            xytext=(0, 7), textcoords="offset points", ha="center", fontsize=8,
        )
    axis.set_xlabel("Hybrid-work intensity")
    axis.set_ylabel(label)
    axis.grid(axis="y", alpha=0.25)
figure.suptitle("National job–home network structure across hybrid-work intensity")
figure.subplots_adjust(top=0.91, hspace=0.36, wspace=0.28)
save_figure(
    figure,
    "figure_2_network_structure_across_intensity_layers",
    FIGURE_MAIN_DIR,
)
atomic_to_csv(
    period_layer_metrics,
    TABLE_MAIN_DIR / "period_layer_network_metric_summary.csv",
)


## 14. Monthly six-layer comparison and gradient stability


In [ ]:
monthly_metric_definitions = {
    "external_flow_share": ("Inter-district flow share", 1.0),
    "mean_external_distance_km": ("Mean external distance (km)", 1.0),
    "network_connections_common_scale": (
        "Network connections at a common flow scale", 1.0
    ),
    "edge_hhi": ("Flow concentration (HHI)", -1.0),
}

figure, axes = plt.subplots(2, 2, figsize=(14.8, 9.2), sharex=True)
for axis, (metric, (label, expected_sign)) in zip(
    axes.flat, monthly_metric_definitions.items()
):
    for band in RECURRENCE_BAND_ORDER:
        frame = layer_metrics.loc[
            layer_metrics["recurrence_band"] == band
        ].sort_values("month")
        axis.plot(
            frame["month"], frame[metric],
            label=band,
            color=INTENSITY_COLOURS[band],
            linewidth=1.5,
            marker="o",
            markersize=2.6,
        )
    axis.set_ylabel(label)
    axis.grid(axis="y", alpha=0.25)
    axis.tick_params(axis="x", rotation=90, labelsize=7)
axes[0, 0].legend(
    title="Workplace recurrence",
    ncol=2,
    frameon=False,
    fontsize=8,
)
figure.suptitle("Monthly network structure across hybrid-work intensity levels")
figure.subplots_adjust(top=0.92, hspace=0.38, wspace=0.25, bottom=0.17)
save_figure(
    figure,
    "figure_3_monthly_network_structure_across_intensity_layers",
    FIGURE_MAIN_DIR,
)

gradient_rows = []
monthly_gradient_details = []
for metric, (label, expected_sign) in monthly_metric_definitions.items():
    for month, frame in layer_metrics.groupby("month"):
        rho, p_value = stats.spearmanr(
            frame["hybrid_work_intensity"], frame[metric], nan_policy="omit"
        )
        monthly_gradient_details.append({
            "month": month,
            "metric": metric,
            "metric_label": label,
            "spearman_rho": rho,
            "p_value": p_value,
            "expected_direction": "positive" if expected_sign > 0 else "negative",
            "same_direction": bool(np.sign(rho) == np.sign(expected_sign)) if np.isfinite(rho) else False,
        })
monthly_gradient_details = pd.DataFrame(monthly_gradient_details)
for (metric, label, expected_direction), frame in monthly_gradient_details.groupby(
    ["metric", "metric_label", "expected_direction"]
):
    gradient_rows.append({
        "metric": metric,
        "metric_label": label,
        "expected_direction": expected_direction,
        "median_monthly_spearman_rho": frame["spearman_rho"].median(),
        "q25_monthly_spearman_rho": frame["spearman_rho"].quantile(0.25),
        "q75_monthly_spearman_rho": frame["spearman_rho"].quantile(0.75),
        "share_months_same_direction": frame["same_direction"].mean(),
        "months": frame["month"].nunique(),
    })
monthly_gradient_summary = pd.DataFrame(gradient_rows)
atomic_to_csv(
    monthly_gradient_details,
    TABLE_APPENDIX_DIR / "monthly_layer_gradient_spearman_details.csv",
)
atomic_to_csv(
    monthly_gradient_summary,
    TABLE_MAIN_DIR / "monthly_layer_gradient_stability.csv",
)
display(monthly_gradient_summary.round(3))


# Part II. District hybrid-work intensity and job–home connections

District hybrid-work intensity is calculated from the full six-layer flow composition. The main panel models compare the same district over time and absorb stable district characteristics and common monthly changes. The reported coefficients describe the association between a 0.10 increase in hybrid-work intensity and each network outcome.


## 15. Outcome-specific district panel models and yearly consistency

Each outcome uses every district-month observation for which that outcome and the primary all-flow hybrid-work intensity are defined. No common complete-case restriction is imposed across the three outcomes.


In [ ]:
OUTCOME_DEFINITIONS = {
    "log1p_partner_coverage": {
        "label": "Partner coverage",
        "raw_column": "partner_coverage",
        "valid_flag": "valid_partner_coverage",
        "transformation": "log1p",
    },
    "log1p_effective_partner_diversity": {
        "label": "Partner diversity",
        "raw_column": "effective_partner_diversity",
        "valid_flag": "valid_partner_diversity",
        "transformation": "log1p",
    },
    "log_mean_external_distance_km": {
        "label": "Mean external distance",
        "raw_column": "mean_external_distance_km",
        "valid_flag": "valid_mean_external_distance",
        "transformation": "log",
    },
}

panel_rows = []
panel_models = {}

for role in ["residential", "employment"]:
    role_data = district_month.loc[
        district_month["role"].eq(role)
        & district_month["valid_intensity"]
    ].copy()

    for outcome, metadata in OUTCOME_DEFINITIONS.items():
        outcome_data = role_data.loc[
            role_data[metadata["valid_flag"]]
        ].copy()

        fitted = fit_two_way_fixed_effect_model(
            outcome_data,
            outcome=outcome,
            predictors=[
                PRIMARY_INTENSITY_COLUMN,
                "log_external_flow",
            ],
        )

        panel_models[(role, outcome, "pooled")] = fitted

        table = coefficient_table(
            fitted,
            model_name="pooled",
        )

        for _, coefficient in table.iterrows():
            row = coefficient.to_dict()
            row.update({
                "role": role,
                "outcome": outcome,
                "outcome_label": metadata["label"],
                "outcome_transformation": metadata["transformation"],
                "year": "pooled",
                "intensity_measure": PRIMARY_INTENSITY_COLUMN,
                "sample_rule": metadata["valid_flag"],
                "n_districts": fitted._analysis_data[
                    "district_id"
                ].nunique(),
                "n_months": fitted._analysis_data["month"].nunique(),
                "within_r_squared": fitted.rsquared,
                "demeaning_iterations": fitted._demeaning_iterations,
            })
            panel_rows.append(row)

        for year in sorted(outcome_data["year"].dropna().unique()):
            year_data = outcome_data.loc[
                outcome_data["year"].eq(year)
            ].copy()

            fitted_year = fit_two_way_fixed_effect_model(
                year_data,
                outcome=outcome,
                predictors=[
                    PRIMARY_INTENSITY_COLUMN,
                    "log_external_flow",
                ],
            )

            panel_models[(role, outcome, int(year))] = fitted_year

            table = coefficient_table(
                fitted_year,
                model_name=str(year),
            )

            for _, coefficient in table.iterrows():
                row = coefficient.to_dict()
                row.update({
                    "role": role,
                    "outcome": outcome,
                    "outcome_label": metadata["label"],
                    "outcome_transformation": metadata["transformation"],
                    "year": int(year),
                    "intensity_measure": PRIMARY_INTENSITY_COLUMN,
                    "sample_rule": metadata["valid_flag"],
                    "n_districts": fitted_year._analysis_data[
                        "district_id"
                    ].nunique(),
                    "n_months": fitted_year._analysis_data[
                        "month"
                    ].nunique(),
                    "within_r_squared": fitted_year.rsquared,
                    "demeaning_iterations": fitted_year._demeaning_iterations,
                })
                panel_rows.append(row)

panel_results = pd.DataFrame(panel_rows)

intensity_results = panel_results.loc[
    panel_results["term"].eq(PRIMARY_INTENSITY_COLUMN)
].copy()

for column, source in [
    ("effect_per_0_1_pct", "estimate"),
    ("effect_per_0_1_ci_low_pct", "ci_low"),
    ("effect_per_0_1_ci_high_pct", "ci_high"),
]:
    intensity_results[column] = 100.0 * np.expm1(
        INTENSITY_EFFECT_INCREMENT
        * intensity_results[source]
    )

atomic_to_csv(
    panel_results,
    TABLE_APPENDIX_DIR
    / "district_intensity_outcome_panel_models_full.csv",
)
atomic_to_csv(
    intensity_results,
    TABLE_MAIN_DIR
    / "district_intensity_outcome_panel_models.csv",
)

display(
    intensity_results[[
        "role",
        "outcome_label",
        "year",
        "effect_per_0_1_pct",
        "effect_per_0_1_ci_low_pct",
        "effect_per_0_1_ci_high_pct",
        "p_value",
        "n",
        "n_districts",
        "within_r_squared",
    ]].round(3)
)


## 16. Coefficient figure for pooled and yearly relationships


In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(14.2, 7.8), sharex=False)
year_order = [2022, 2023, 2024, "pooled"]
year_labels = ["2022", "2023", "2024", "Pooled"]
for row_index, role in enumerate(["residential", "employment"]):
    for column_index, (outcome, metadata) in enumerate(OUTCOME_DEFINITIONS.items()):
        axis = axes[row_index, column_index]
        frame = intensity_results.loc[
            (intensity_results["role"] == role)
            & (intensity_results["outcome"] == outcome)
        ].copy()
        frame["year_order"] = frame["year"].map({2022: 0, 2023: 1, 2024: 2, "pooled": 3})
        frame = frame.sort_values("year_order")
        y = np.arange(len(frame))
        axis.errorbar(
            frame["effect_per_0_1_pct"],
            y,
            xerr=[
                frame["effect_per_0_1_pct"] - frame["effect_per_0_1_ci_low_pct"],
                frame["effect_per_0_1_ci_high_pct"] - frame["effect_per_0_1_pct"],
            ],
            fmt="o",
            capsize=3,
        )
        axis.axvline(0, color="black", linewidth=0.8)
        axis.set_yticks(y, frame["year"].astype(str))
        axis.set_title(metadata["label"])
        axis.set_xlabel("Associated multiplicative change for a 0.10 increase in intensity (%)")
        if column_index == 0:
            axis.set_ylabel(ROLE_LABELS[role])
        axis.grid(axis="x", alpha=0.25)
figure.suptitle("Hybrid-work intensity and district job–home network outcomes")
figure.subplots_adjust(top=0.90, hspace=0.48, wspace=0.34)
save_figure(
    figure,
    "figure_4_intensity_and_district_network_outcomes",
    FIGURE_MAIN_DIR,
)


## 17. Within-district partial correlations and clustered inference

This section produces manuscript Table 2. The correlations are calculated after removing district effects, common monthly conditions, and external-flow volume. District-clustered inference allows serial dependence within spatial units.

In [ ]:
# ---------------------------------------------------------------------
# Within-district partial correlations for manuscript Table 2
# ---------------------------------------------------------------------
# This table complements the fixed-effects coefficient figure. It reports
# correlations on a common [-1, 1] scale after removing:
#   1. district fixed effects,
#   2. month fixed effects, and
#   3. log external-flow volume.
#
# The same transformations used by the panel models are retained. A common
# complete-case sample is used within each role so that every entry in a
# role-specific matrix is directly comparable.

# The cell can be run after the full notebook or independently in a fresh
# kernel. When the panel is not already in memory, it reads the cached panel
# table produced by the original workflow.
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

try:
    from IPython.display import display
except ImportError:
    display = print


if "PRIMARY_INTENSITY_COLUMN" not in globals():
    PRIMARY_INTENSITY_COLUMN = "hybrid_intensity_all_flows"

if "ROLE_LABELS" not in globals():
    ROLE_LABELS = {
        "residential": "Residential role",
        "employment": "Employment role",
    }

if "FE_DEMEAN_TOLERANCE" not in globals():
    FE_DEMEAN_TOLERANCE = 1e-10

if "FE_DEMEAN_MAX_ITER" not in globals():
    FE_DEMEAN_MAX_ITER = 500

if "TABLE_MAIN_DIR" not in globals():
    _partial_corr_project_root = Path(
        os.environ.get("HYBRID_WORK_PROJECT_ROOT", Path.cwd())
    ).expanduser().resolve()
    TABLE_MAIN_DIR = (
        _partial_corr_project_root
        / "05_analysis"
        / "13_hybrid_network_article"
        / "01_intensity_gradient_analysis_v4_4_20260724"
        / "tables_main"
    )
else:
    TABLE_MAIN_DIR = Path(TABLE_MAIN_DIR)


if "require_columns" not in globals():
    def require_columns(frame, required, frame_name):
        missing = sorted(set(required) - set(frame.columns))
        if missing:
            raise KeyError(
                f"{frame_name} lacks required fields: {missing}. "
                f"Available fields: {frame.columns.tolist()}"
            )


if "atomic_to_csv" not in globals():
    def atomic_to_csv(frame, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + ".tmp")
        frame.to_csv(
            temporary,
            index=False,
            encoding="utf-8-sig",
        )
        temporary.replace(path)


if "iterative_two_way_demean" not in globals():
    def iterative_two_way_demean(
        frame,
        columns,
        entity_column,
        time_column,
    ):
        """Alternating-projection demeaning for entity and time effects."""
        transformed = frame[columns].astype(float).copy()
        for iteration in range(FE_DEMEAN_MAX_ITER):
            previous = transformed.to_numpy(copy=True)
            transformed = transformed - transformed.groupby(
                frame[entity_column]
            ).transform("mean")
            transformed = transformed - transformed.groupby(
                frame[time_column]
            ).transform("mean")
            difference = np.nanmax(
                np.abs(transformed.to_numpy() - previous)
            )
            if (
                np.isfinite(difference)
                and difference < FE_DEMEAN_TOLERANCE
            ):
                return transformed, iteration + 1
        raise RuntimeError(
            "Two-way demeaning did not converge within "
            f"{FE_DEMEAN_MAX_ITER} iterations."
        )


if "district_month" not in globals():
    _partial_corr_panel_path = (
        TABLE_MAIN_DIR
        / "district_month_role_intensity_metrics.parquet"
    )
    if not _partial_corr_panel_path.exists():
        raise FileNotFoundError(
            "The district-month panel is not in memory and the cached "
            "panel table was not found at:\n"
            f"{_partial_corr_panel_path}\n\n"
            "Run the notebook through the district-month export cell first, "
            "set HYBRID_WORK_PROJECT_ROOT, or define TABLE_MAIN_DIR as "
            "the directory containing "
            "district_month_role_intensity_metrics.parquet."
        )
    district_month = pd.read_parquet(_partial_corr_panel_path)
    print("Loaded district-month panel:", _partial_corr_panel_path)


PARTIAL_CORRELATION_VARIABLES = {
    PRIMARY_INTENSITY_COLUMN: "Hybrid-work intensity",
    "log1p_partner_coverage": "Partner coverage",
    "log1p_effective_partner_diversity": "Partner diversity",
    "log_mean_external_distance_km": "Mean external distance",
}
PARTIAL_CORRELATION_CONTROLS = ["log_external_flow"]
PARTIAL_CORRELATION_ROLE_ORDER = ["residential", "employment"]


def residualize_partial_correlation_variables(
    frame,
    target_columns,
    control_columns,
    entity_column="district_id",
    time_column="month",
):
    """Residualize targets with respect to two-way fixed effects and controls."""
    required = [
        entity_column,
        time_column,
        *target_columns,
        *control_columns,
    ]
    require_columns(frame, required, "partial-correlation input")

    data = (
        frame[required]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )
    if data.empty:
        raise ValueError("No complete observations for partial correlations.")

    numeric_columns = [*target_columns, *control_columns]
    within, iterations = iterative_two_way_demean(
        data,
        numeric_columns,
        entity_column=entity_column,
        time_column=time_column,
    )

    controls = within[control_columns].to_numpy(dtype=float)
    residuals = pd.DataFrame(index=data.index)

    for column in target_columns:
        values = within[column].to_numpy(dtype=float)
        if controls.shape[1] == 0:
            residual = values
        else:
            coefficients, *_ = np.linalg.lstsq(
                controls,
                values,
                rcond=None,
            )
            residual = values - controls @ coefficients
        residuals[column] = residual

    identifiers = data[[entity_column, time_column]].copy()
    return identifiers.join(residuals), iterations


def significance_marker(p_value):
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""


def p_value_label(p_value):
    if p_value < 0.001:
        return "<0.001"
    return f"{p_value:.3f}"


def clustered_partial_correlation_inference(
    residual_data,
    variable_1,
    variable_2,
    cluster_column="district_id",
):
    """Partial r with district-clustered robust inference.

    The two residual series are standardized before fitting a no-intercept
    regression. The fitted slope therefore equals their Pearson correlation.
    District-clustered covariance permits heteroskedasticity and serial
    dependence within districts.
    """
    required = [cluster_column, variable_1, variable_2]
    data = (
        residual_data[required]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )
    if data.empty:
        raise ValueError(
            f"No valid residual pairs for {variable_1} and {variable_2}."
        )

    x = data[variable_1].to_numpy(dtype=float)
    y = data[variable_2].to_numpy(dtype=float)
    x_sd = float(np.std(x, ddof=0))
    y_sd = float(np.std(y, ddof=0))
    if x_sd <= 0 or y_sd <= 0:
        raise ValueError(
            f"Zero residual variation for {variable_1} or {variable_2}."
        )

    x_standardized = (x - x.mean()) / x_sd
    y_standardized = (y - y.mean()) / y_sd
    groups = data[cluster_column]

    cross_product = float(
        np.dot(x_standardized, x_standardized)
    )
    partial_r = float(
        np.dot(x_standardized, y_standardized)
        / cross_product
    )
    direct_r = float(np.corrcoef(x, y)[0, 1])
    if not np.isclose(partial_r, direct_r, atol=1e-10):
        raise RuntimeError(
            "The standardized residual slope does not equal the "
            "partial correlation."
        )

    regression_residual = (
        y_standardized
        - partial_r * x_standardized
    )
    cluster_scores = pd.DataFrame({
        cluster_column: groups.to_numpy(),
        "score": x_standardized * regression_residual,
    }).groupby(cluster_column, sort=False)["score"].sum()

    n_clusters = int(data[cluster_column].nunique())
    if n_clusters < 2:
        raise ValueError(
            "At least two districts are required for clustered inference."
        )

    # One-regressor, no-intercept CR1 cluster covariance. With one fitted
    # parameter, the usual observation-level correction equals one; the
    # remaining finite-sample adjustment is G/(G-1).
    cluster_correction = n_clusters / (n_clusters - 1)
    cluster_variance = (
        cluster_correction
        * float(np.square(cluster_scores).sum())
        / (cross_product ** 2)
    )
    cluster_robust_se = float(
        np.sqrt(max(cluster_variance, 0.0))
    )
    inference_df = n_clusters - 1
    test_statistic = (
        partial_r / cluster_robust_se
        if cluster_robust_se > 0
        else np.nan
    )
    p_value = (
        float(
            2.0
            * stats.t.sf(
                abs(test_statistic),
                df=inference_df,
            )
        )
        if np.isfinite(test_statistic)
        else np.nan
    )
    critical_value = float(
        stats.t.ppf(0.975, df=inference_df)
    )
    confidence = np.array([
        partial_r - critical_value * cluster_robust_se,
        partial_r + critical_value * cluster_robust_se,
    ])

    return {
        "partial_correlation": partial_r,
        "cluster_robust_se": cluster_robust_se,
        "test_statistic": float(test_statistic),
        "p_value": p_value,
        "p_value_label": p_value_label(p_value),
        "significance": significance_marker(p_value),
        "ci_low": float(confidence[0]),
        "ci_high": float(confidence[1]),
        "confidence_level": 0.95,
        "n_observations": int(len(data)),
        "n_districts": n_clusters,
        "inference_df": float(inference_df),
        "covariance_estimator": (
            "district-clustered robust covariance with finite-sample "
            "correction"
        ),
    }


partial_correlation_panels = {}
partial_correlation_tidy_parts = []
partial_correlation_inference_parts = []
partial_correlation_intensity_rows = []
partial_correlation_sample_rows = []

target_columns = list(PARTIAL_CORRELATION_VARIABLES)
intensity_label = PARTIAL_CORRELATION_VARIABLES[
    PRIMARY_INTENSITY_COLUMN
]

for role in PARTIAL_CORRELATION_ROLE_ORDER:
    role_data = district_month.loc[
        district_month["role"].eq(role)
        & district_month["valid_intensity"]
    ].copy()

    residual_data, demeaning_iterations = (
        residualize_partial_correlation_variables(
            role_data,
            target_columns=target_columns,
            control_columns=PARTIAL_CORRELATION_CONTROLS,
        )
    )

    correlation_matrix = (
        residual_data[target_columns]
        .corr(method="pearson")
        .rename(
            index=PARTIAL_CORRELATION_VARIABLES,
            columns=PARTIAL_CORRELATION_VARIABLES,
        )
    )
    partial_correlation_panels[role] = correlation_matrix

    tidy = (
        correlation_matrix
        .rename_axis(index="variable_1", columns="variable_2")
        .stack()
        .rename("partial_correlation")
        .reset_index()
    )
    variable_order = {
        label: index
        for index, label in enumerate(
            PARTIAL_CORRELATION_VARIABLES.values()
        )
    }
    tidy["variable_1_order"] = tidy["variable_1"].map(variable_order)
    tidy["variable_2_order"] = tidy["variable_2"].map(variable_order)
    tidy = tidy.loc[
        tidy["variable_1_order"] >= tidy["variable_2_order"]
    ].copy()
    tidy.insert(0, "role", role)
    tidy["n_observations"] = len(residual_data)
    tidy["n_districts"] = residual_data["district_id"].nunique()
    tidy["n_months"] = residual_data["month"].nunique()
    tidy["demeaning_iterations"] = demeaning_iterations
    partial_correlation_tidy_parts.append(tidy)

    role_inference_rows = []
    for row_index in range(1, len(target_columns)):
        variable_1 = target_columns[row_index]
        variable_1_label = PARTIAL_CORRELATION_VARIABLES[
            variable_1
        ]

        for column_index in range(row_index):
            variable_2 = target_columns[column_index]
            variable_2_label = PARTIAL_CORRELATION_VARIABLES[
                variable_2
            ]

            inference = clustered_partial_correlation_inference(
                residual_data,
                variable_1=variable_1,
                variable_2=variable_2,
            )
            inference.update({
                "role": role,
                "variable_1": variable_1_label,
                "variable_2": variable_2_label,
                "variable_1_column": variable_1,
                "variable_2_column": variable_2,
                "n_months": int(
                    residual_data["month"].nunique()
                ),
                "demeaning_iterations": int(
                    demeaning_iterations
                ),
            })
            role_inference_rows.append(inference)

            if variable_2 == PRIMARY_INTENSITY_COLUMN:
                intensity_result = dict(inference)
                intensity_result["outcome"] = variable_1_label
                partial_correlation_intensity_rows.append(
                    intensity_result
                )

    partial_correlation_inference_parts.append(
        pd.DataFrame(role_inference_rows)
    )

    partial_correlation_sample_rows.append({
        "role": role,
        "n_observations": int(len(residual_data)),
        "n_districts": int(
            residual_data["district_id"].nunique()
        ),
        "n_months": int(residual_data["month"].nunique()),
        "demeaning_iterations": int(demeaning_iterations),
        "controls": (
            "district fixed effects; month fixed effects; "
            "log external-flow volume"
        ),
    })


district_partial_correlations = pd.concat(
    partial_correlation_tidy_parts,
    ignore_index=True,
).drop(
    columns=["variable_1_order", "variable_2_order"]
)

district_partial_correlation_inference = pd.concat(
    partial_correlation_inference_parts,
    ignore_index=True,
)

table_2_partial_correlations_long = pd.DataFrame(
    partial_correlation_intensity_rows
)
table_2_partial_correlations = (
    table_2_partial_correlations_long
    .pivot(
        index="outcome",
        columns="role",
        values="partial_correlation",
    )
    .reindex([
        "Partner coverage",
        "Partner diversity",
        "Mean external distance",
    ])
    .rename(columns={
        "residential": "Residential role",
        "employment": "Employment role",
    })
    .reset_index()
)
table_2_partial_correlations_full = (
    table_2_partial_correlations_long[[
        "role",
        "outcome",
        "partial_correlation",
        "cluster_robust_se",
        "ci_low",
        "ci_high",
        "test_statistic",
        "p_value",
        "p_value_label",
        "significance",
        "n_observations",
        "n_districts",
        "n_months",
        "inference_df",
        "covariance_estimator",
    ]]
    .copy()
)
table_2_partial_correlations_full["role"] = (
    table_2_partial_correlations_full["role"].map(ROLE_LABELS)
)
table_2_partial_correlations_full["95% CI"] = (
    "["
    + table_2_partial_correlations_full["ci_low"]
    .map(lambda value: f"{value:.3f}")
    + ", "
    + table_2_partial_correlations_full["ci_high"]
    .map(lambda value: f"{value:.3f}")
    + "]"
)
table_2_partial_correlations_publication = (
    table_2_partial_correlations_full[[
        "role",
        "outcome",
        "partial_correlation",
        "cluster_robust_se",
        "95% CI",
        "test_statistic",
        "p_value_label",
        "significance",
        "n_observations",
        "n_districts",
        "n_months",
    ]]
    .rename(columns={
        "role": "Role",
        "outcome": "Outcome",
        "partial_correlation": "Partial r",
        "cluster_robust_se": "Cluster-robust SE",
        "test_statistic": "t",
        "p_value_label": "p",
        "significance": "Sig.",
        "n_observations": "N",
        "n_districts": "Districts",
        "n_months": "Months",
    })
)
partial_correlation_sample_diagnostics = pd.DataFrame(
    partial_correlation_sample_rows
)

atomic_to_csv(
    district_partial_correlations,
    TABLE_MAIN_DIR
    / "district_within_partial_correlation_matrices.csv",
)
atomic_to_csv(
    district_partial_correlation_inference,
    TABLE_MAIN_DIR
    / "district_within_partial_correlations_clustered_inference.csv",
)
atomic_to_csv(
    table_2_partial_correlations_long,
    TABLE_MAIN_DIR
    / "district_intensity_outcome_partial_correlations_long.csv",
)
atomic_to_csv(
    table_2_partial_correlations,
    TABLE_MAIN_DIR
    / "table_2_district_intensity_outcome_partial_correlations.csv",
)
atomic_to_csv(
    table_2_partial_correlations_full,
    TABLE_MAIN_DIR
    / "table_2_district_intensity_outcome_partial_correlations_full.csv",
)
atomic_to_csv(
    table_2_partial_correlations_publication,
    TABLE_MAIN_DIR
    / "table_2_district_intensity_outcome_partial_correlations_publication.csv",
)
atomic_to_csv(
    partial_correlation_sample_diagnostics,
    TABLE_MAIN_DIR
    / "district_partial_correlation_sample_diagnostics.csv",
)

for role in PARTIAL_CORRELATION_ROLE_ORDER:
    print(f"\n{ROLE_LABELS[role]}: within-district partial correlations")
    display(partial_correlation_panels[role].round(3))

print("\nCompact manuscript Table 2")
display(table_2_partial_correlations.round(3))

print("\nComplete manuscript Table 2 with district-clustered inference")
display(
    table_2_partial_correlations_publication.round({
        "Partial r": 3,
        "Cluster-robust SE": 3,
        "t": 2,
    })
)

print("\nSample diagnostics")
display(partial_correlation_sample_diagnostics)


## 18. Full-period district indicators and existing onsite network position

Period maps are recalculated from the complete 2022–2024 network. They use every district available for each measure and do not inherit the common regression sample.


In [ ]:
# ---------------------------------------------------------------------
# Full-period network metrics
# ---------------------------------------------------------------------
period_full_edges = (
    period_layer_edges.groupby(
        ["origin", "destination"],
        as_index=False,
    )
    .agg(
        weight=("weight", "sum"),
        distance_km=("distance_km", "first"),
    )
)

all_district_role_grid = pd.MultiIndex.from_product(
    [
        sorted(all_network_ids),
        ["residential", "employment"],
    ],
    names=["district_id", "role"],
).to_frame(index=False)

period_metric_parts = []
period_partner_parts = []

for role in ["residential", "employment"]:
    metrics = node_role_metrics(
        period_full_edges,
        role,
    )
    period_metric_parts.append(metrics)

    pairs = external_partner_pairs(
        period_full_edges,
        role,
    )
    pairs["role"] = role
    period_partner_parts.append(pairs)

period_network_metrics = pd.concat(
    period_metric_parts,
    ignore_index=True,
)

period_partner_pairs = pd.concat(
    period_partner_parts,
    ignore_index=True,
)

period_network_metrics = all_district_role_grid.merge(
    period_network_metrics,
    on=["district_id", "role"],
    how="left",
    validate="one_to_one",
)

zero_fill_columns = [
    "total_flow",
    "network_flow_share",
    "partner_count_raw",
    "local_flow",
    "external_flow",
    "external_partner_count",
    "partner_entropy",
    "effective_partner_diversity",
    "normalized_partner_entropy",
    "largest_partner_share",
]

for column in zero_fill_columns:
    if column in period_network_metrics.columns:
        period_network_metrics[column] = pd.to_numeric(
            period_network_metrics[column],
            errors="coerce",
        ).fillna(0.0)

period_network_metrics["partner_coverage"] = (
    period_network_metrics["external_partner_count"]
)
period_network_metrics["partner_diversity"] = (
    period_network_metrics["effective_partner_diversity"]
)
period_network_metrics["has_any_flow"] = (
    period_network_metrics["total_flow"] > 0
)
period_network_metrics["has_external_relation"] = (
    period_network_metrics["external_flow"] > 0
)

period_network_metrics.loc[
    ~period_network_metrics["has_external_relation"],
    "mean_external_distance_km",
] = np.nan


# ---------------------------------------------------------------------
# Full-period standardised coverage, retained as robustness information
# ---------------------------------------------------------------------
eligible_period_external_flow = period_network_metrics.loc[
    period_network_metrics["has_external_relation"]
    & period_network_metrics["external_partner_count"].ge(
        MIN_EXTERNAL_PARTNERS
    ),
    "external_flow",
].dropna()

if eligible_period_external_flow.empty:
    period_partner_draws = 1
else:
    period_partner_draws = int(max(
        1,
        min(
            NODE_RAREFACTION_DRAW_CAP,
            math.floor(
                float(
                    eligible_period_external_flow.quantile(0.05)
                )
            ),
        ),
    ))

period_pairs_working = period_partner_pairs.copy()

if not period_pairs_working.empty:
    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore",
    ):
        period_pairs_working["occupied_probability"] = -np.expm1(
            period_partner_draws
            * np.log1p(
                -period_pairs_working["partner_probability"]
            )
        )

    period_rarefied = (
        period_pairs_working.groupby(
            ["district_id", "role"],
            as_index=False,
        )
        .agg(
            partner_coverage_rarefied_period=(
                "occupied_probability",
                "sum",
            )
        )
    )
else:
    period_rarefied = pd.DataFrame(
        columns=[
            "district_id",
            "role",
            "partner_coverage_rarefied_period",
        ]
    )

period_network_metrics = period_network_metrics.merge(
    period_rarefied,
    on=["district_id", "role"],
    how="left",
    validate="one_to_one",
)

valid_rarefied_support = (
    period_network_metrics["external_flow"].ge(
        period_partner_draws
    )
    & period_network_metrics["external_partner_count"].ge(
        MIN_EXTERNAL_PARTNERS
    )
)

period_network_metrics.loc[
    ~valid_rarefied_support,
    "partner_coverage_rarefied_period",
] = np.nan


# ---------------------------------------------------------------------
# Period intensity from every available month
# ---------------------------------------------------------------------
period_intensity_rows = []

for (district_id, role), group in district_month.groupby(
    ["district_id", "role"],
    sort=False,
):
    total_flow = group["total_flow"].fillna(0).to_numpy(dtype=float)
    external_flow = group["external_flow"].fillna(0).to_numpy(dtype=float)

    all_intensity = pd.to_numeric(
        group["hybrid_intensity_all_flows"],
        errors="coerce",
    ).to_numpy(dtype=float)

    external_intensity = pd.to_numeric(
        group["hybrid_intensity_external"],
        errors="coerce",
    ).to_numpy(dtype=float)

    valid_all = np.isfinite(all_intensity) & (total_flow > 0)
    valid_external = (
        np.isfinite(external_intensity)
        & (external_flow > 0)
    )

    period_intensity_rows.append({
        "district_id": district_id,
        "role": role,
        "valid_months_intensity_all": int(valid_all.sum()),
        "valid_months_intensity_external": int(
            valid_external.sum()
        ),
        "hybrid_intensity_all_flows": (
            float(
                np.average(
                    all_intensity[valid_all],
                    weights=total_flow[valid_all],
                )
            )
            if valid_all.any()
            else np.nan
        ),
        "hybrid_intensity_external": (
            float(
                np.average(
                    external_intensity[valid_external],
                    weights=external_flow[valid_external],
                )
            )
            if valid_external.any()
            else np.nan
        ),
    })

period_intensity = pd.DataFrame(period_intensity_rows)


# Outcome-specific monthly observation counts for documentation.
valid_month_counts = (
    district_month.groupby(
        ["district_id", "role"],
        as_index=False,
    )
    .agg(
        valid_months_partner_coverage=(
            "valid_partner_coverage",
            "sum",
        ),
        valid_months_partner_diversity=(
            "valid_partner_diversity",
            "sum",
        ),
        valid_months_mean_external_distance=(
            "valid_mean_external_distance",
            "sum",
        ),
    )
)


# ---------------------------------------------------------------------
# Existing onsite network position with structural zeros retained
# ---------------------------------------------------------------------
onsite_period_edges = (
    period_layer_edges.loc[
        period_layer_edges["recurrence_band"].isin(
            HIGH_ATTENDANCE_BANDS
        )
    ]
    .groupby(
        ["origin", "destination"],
        as_index=False,
    )
    .agg(
        weight=("weight", "sum"),
        distance_km=("distance_km", "first"),
    )
)

position_parts = []

for role, node_column in [
    ("residential", "origin"),
    ("employment", "destination"),
]:
    positive_position = (
        onsite_period_edges.groupby(
            node_column,
            as_index=False,
        )["weight"]
        .sum()
        .rename(
            columns={
                node_column: "district_id",
                "weight": "onsite_strength",
            }
        )
    )
    positive_position["role"] = role

    position = all_district_role_grid.loc[
        all_district_role_grid["role"].eq(role)
    ].merge(
        positive_position,
        on=["district_id", "role"],
        how="left",
        validate="one_to_one",
    )

    position["onsite_strength"] = pd.to_numeric(
        position["onsite_strength"],
        errors="coerce",
    ).fillna(0.0)

    position["onsite_network_position"] = 0.0
    positive = position["onsite_strength"] > 0

    if positive.any():
        position.loc[
            positive,
            "onsite_network_position",
        ] = position.loc[
            positive,
            "onsite_strength",
        ].rank(
            method="average",
            pct=True,
        )

    position_parts.append(position)

onsite_position = pd.concat(
    position_parts,
    ignore_index=True,
)


# ---------------------------------------------------------------------
# Assemble the full descriptive district-role table
# ---------------------------------------------------------------------
period_district = (
    period_network_metrics
    .merge(
        period_intensity,
        on=["district_id", "role"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        valid_month_counts,
        on=["district_id", "role"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        onsite_position,
        on=["district_id", "role"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        ghs_profile,
        on="district_id",
        how="left",
        validate="many_to_one",
    )
)

period_district["province_code"] = period_district[
    "district_id"
].map(province_code_from_district)

period_district["primary_intensity"] = period_district[
    PRIMARY_INTENSITY_COLUMN
]

period_district["log1p_partner_coverage"] = np.log1p(
    period_district["partner_coverage"].clip(lower=0)
)
period_district["log1p_effective_partner_diversity"] = np.log1p(
    period_district["effective_partner_diversity"].clip(lower=0)
)
# Compatibility alias retained for Notebook 02 and earlier map inputs.
period_district["log1p_partner_diversity"] = period_district[
    "log1p_effective_partner_diversity"
]
period_district["log_mean_external_distance_km"] = np.where(
    period_district["mean_external_distance_km"] > 0,
    np.log(period_district["mean_external_distance_km"]),
    np.nan,
)
period_district["log_population"] = pd.to_numeric(
    period_district["log_population"],
    errors="coerce",
)
period_district["log_external_flow"] = np.log1p(
    period_district["external_flow"].clip(lower=0)
)
period_district["log_total_flow"] = np.log1p(
    period_district["total_flow"].clip(lower=0)
)


if RUN_PERIOD_SPECTRAL_CENTRALITY:
    spectral = spectral_centrality(
        onsite_period_edges
    )

    residential_spectral = spectral[
        ["district_id", "hub_score"]
    ].copy()
    residential_spectral["role"] = "residential"
    residential_spectral = residential_spectral.rename(
        columns={
            "hub_score": "onsite_spectral_position"
        }
    )

    employment_spectral = spectral[
        ["district_id", "authority_score"]
    ].copy()
    employment_spectral["role"] = "employment"
    employment_spectral = employment_spectral.rename(
        columns={
            "authority_score": "onsite_spectral_position"
        }
    )

    spectral_role = pd.concat(
        [
            residential_spectral,
            employment_spectral,
        ],
        ignore_index=True,
    )

    period_district = period_district.merge(
        spectral_role,
        on=["district_id", "role"],
        how="left",
        validate="one_to_one",
    )

    period_district[
        "onsite_spectral_position"
    ] = pd.to_numeric(
        period_district["onsite_spectral_position"],
        errors="coerce",
    ).fillna(0.0)


period_ghs_match_share = float(
    period_district.groupby(
        "district_id"
    )["settlement_urbanity"]
    .first()
    .notna()
    .mean()
)

period_ghs_audit = pd.DataFrame([{
    "period_rows": int(len(period_district)),
    "period_unique_districts": int(
        period_district["district_id"].nunique()
    ),
    "unique_district_match_share": period_ghs_match_share,
    "period_partner_draws": period_partner_draws,
}])

atomic_to_csv(
    period_ghs_audit,
    LOG_DIR / "period_district_ghs_merge_audit.csv",
)
display(period_ghs_audit)

if (
    REQUIRE_GHS_PROFILE
    and period_ghs_match_share < MIN_GHS_MATCH_SHARE
):
    raise RuntimeError(
        "The period-level district table matched too few districts to GHSL: "
        f"{period_ghs_match_share:.2%} < "
        f"{MIN_GHS_MATCH_SHARE:.2%}."
    )


period_variable_coverage = pd.DataFrame([
    {
        "role": role,
        "variable": variable,
        "valid_districts": int(
            period_district.loc[
                period_district["role"].eq(role),
                variable,
            ].notna().sum()
        ),
        "total_districts": int(
            period_district.loc[
                period_district["role"].eq(role)
            ].shape[0]
        ),
        "coverage_share": float(
            period_district.loc[
                period_district["role"].eq(role),
                variable,
            ].notna().mean()
        ),
    }
    for role in ["residential", "employment"]
    for variable in [
        "hybrid_intensity_all_flows",
        "hybrid_intensity_external",
        "partner_coverage",
        "partner_diversity",
        "mean_external_distance_km",
        "settlement_urbanity",
        "onsite_network_position",
    ]
])

atomic_to_csv(
    period_variable_coverage,
    TABLE_MAIN_DIR / "period_variable_coverage_diagnostics.csv",
)


# Main and compatibility exports.
atomic_to_parquet(
    period_district,
    TABLE_MAIN_DIR
    / "district_period_role_all_available_metrics.parquet",
)
atomic_to_parquet(
    period_district,
    MAP_INPUT_DIR
    / "district_period_role_all_available_metrics_map_ready.parquet",
)
atomic_to_parquet(
    period_district,
    TABLE_MAIN_DIR
    / "district_period_role_intensity_metrics.parquet",
)
atomic_to_parquet(
    period_district,
    MAP_INPUT_DIR
    / "district_period_role_intensity_metrics_map_ready.parquet",
)

display(period_variable_coverage)
display(period_district.head())


# Part III. Urbanity, existing network position, and district outcomes

The period models hold hybrid-work intensity constant while estimating the associations of settlement urbanity and existing onsite network position with each outcome. Interaction terms then test whether these two district characteristics change the intensity–outcome relationship.


## 19. Outcome-specific context and network-position models

The three context models no longer share a common complete-case outcome sample. Each model retains every district for which its own outcome, hybrid-work intensity, urbanity, network position and controls are available.


In [ ]:
CONTEXT_OUTCOMES = {
    "log1p_partner_coverage": {
        "label": "Partner coverage",
        "raw_column": "partner_coverage",
        "backtransform_offset": 1.0,
    },
    "log1p_effective_partner_diversity": {
        "label": "Partner diversity",
        "raw_column": "partner_diversity",
        "backtransform_offset": 1.0,
    },
    "log_mean_external_distance_km": {
        "label": "Mean external distance",
        "raw_column": "mean_external_distance_km",
        "backtransform_offset": 0.0,
    },
}

context_model_rows = []
context_models = {}
context_model_data = {}

intensity_centered = f"{PRIMARY_INTENSITY_COLUMN}_c"

for role in ["residential", "employment"]:
    role_data = period_district.loc[
        period_district["role"].eq(role)
    ].copy()

    for outcome, metadata in CONTEXT_OUTCOMES.items():
        required_columns = [
            outcome,
            PRIMARY_INTENSITY_COLUMN,
            "settlement_urbanity",
            "onsite_network_position",
            "log_population",
            "log_external_flow",
            "province_code",
        ]

        require_columns(
            role_data,
            required_columns,
            f"period context input ({role}, {outcome})",
        )
        model_data = (
            role_data
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=required_columns)
            .copy()
        )

        if len(model_data) == 0:
            raise RuntimeError(
                f"No valid context-model observations for {role}, {outcome}."
            )

        for column in [
            PRIMARY_INTENSITY_COLUMN,
            "settlement_urbanity",
            "onsite_network_position",
            "log_external_flow",
        ]:
            model_data[f"{column}_c"] = (
                model_data[column]
                - model_data[column].mean()
            )

        formula = (
            f"{outcome} ~ {intensity_centered} "
            "+ settlement_urbanity_c "
            "+ onsite_network_position_c "
            f"+ {intensity_centered}:settlement_urbanity_c "
            f"+ {intensity_centered}:onsite_network_position_c "
            "+ log_population "
            "+ log_external_flow_c "
            "+ C(province_code)"
        )

        fitted = smf.ols(
            formula,
            data=model_data,
        ).fit(cov_type="HC3")

        context_models[(role, outcome)] = fitted
        context_model_data[(role, outcome)] = model_data

        table = coefficient_table(
            fitted,
            model_name=f"{role}_{outcome}",
        )

        for _, coefficient in table.iterrows():
            row = coefficient.to_dict()
            row.update({
                "role": role,
                "outcome": outcome,
                "outcome_label": metadata["label"],
                "intensity_measure": PRIMARY_INTENSITY_COLUMN,
                "adjusted_r_squared": fitted.rsquared_adj,
                "n_districts": model_data[
                    "district_id"
                ].nunique(),
            })
            context_model_rows.append(row)


context_model_results = pd.DataFrame(
    context_model_rows
)

core_terms = [
    intensity_centered,
    "settlement_urbanity_c",
    "onsite_network_position_c",
    f"{intensity_centered}:settlement_urbanity_c",
    f"{intensity_centered}:onsite_network_position_c",
]

context_core_results = context_model_results.loc[
    context_model_results["term"].isin(
        core_terms
    )
].copy()

atomic_to_csv(
    context_core_results,
    TABLE_MAIN_DIR
    / "district_context_intensity_interaction_models.csv",
)
atomic_to_csv(
    context_model_results,
    TABLE_APPENDIX_DIR
    / "district_context_intensity_interaction_models_full.csv",
)

display(
    context_core_results[[
        "role",
        "outcome_label",
        "term",
        "estimate",
        "robust_se",
        "p_value",
        "adjusted_r_squared",
        "n_districts",
    ]].round(4)
)


## 20. Conditional slopes and formal moderation contrasts

The period interaction models remain the main contextual analysis. This section makes their inferential content explicit. It reports the two interaction terms, the intensity slope at the 20th, 50th, and 80th percentiles of each moderator, the 80th-minus-20th percentile slope contrast, and formal employment-versus-residential interaction tests. The other moderator is held at its sample mean when conditional slopes are evaluated.

In [ ]:
# ---------------------------------------------------------------------
# Formal inference for contextual moderation
# ---------------------------------------------------------------------
def context_significance_marker(p_value):
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""


def context_p_value_label(p_value):
    if p_value < 0.001:
        return "<0.001"
    return f"{p_value:.3f}"


def linear_combination_inference(
    fitted,
    term_weights,
    confidence_level=0.95,
):
    """Inference for an arbitrary linear combination of model terms."""
    parameter_names = list(fitted.params.index)
    missing = sorted(set(term_weights) - set(parameter_names))
    if missing:
        raise KeyError(
            "The fitted model lacks terms required by the contrast: "
            f"{missing}"
        )

    contrast = np.zeros(len(parameter_names), dtype=float)
    positions = {
        term: index
        for index, term in enumerate(parameter_names)
    }
    for term, weight in term_weights.items():
        contrast[positions[term]] = float(weight)

    parameters = fitted.params.to_numpy(dtype=float)
    covariance = fitted.cov_params().to_numpy(dtype=float)
    estimate = float(contrast @ parameters)
    variance = float(contrast @ covariance @ contrast)
    robust_se = float(np.sqrt(max(variance, 0.0)))

    use_t = bool(getattr(fitted, "use_t", False))
    if use_t:
        inference_df = float(fitted.df_resid)
        critical_value = float(
            stats.t.ppf(
                0.5 + confidence_level / 2.0,
                df=inference_df,
            )
        )
        p_value = (
            float(
                2.0
                * stats.t.sf(
                    abs(estimate / robust_se),
                    df=inference_df,
                )
            )
            if robust_se > 0
            else np.nan
        )
        inference_distribution = "t"
    else:
        inference_df = np.nan
        critical_value = float(
            stats.norm.ppf(
                0.5 + confidence_level / 2.0
            )
        )
        p_value = (
            float(
                2.0
                * stats.norm.sf(
                    abs(estimate / robust_se)
                )
            )
            if robust_se > 0
            else np.nan
        )
        inference_distribution = "normal"

    test_statistic = (
        float(estimate / robust_se)
        if robust_se > 0
        else np.nan
    )
    ci_low = estimate - critical_value * robust_se
    ci_high = estimate + critical_value * robust_se

    return {
        "estimate_log": estimate,
        "robust_se": robust_se,
        "test_statistic": test_statistic,
        "p_value": p_value,
        "p_value_label": context_p_value_label(p_value),
        "significance": context_significance_marker(p_value),
        "ci_low_log": float(ci_low),
        "ci_high_log": float(ci_high),
        "confidence_level": confidence_level,
        "inference_distribution": inference_distribution,
        "inference_df": inference_df,
    }


def find_interaction_term(parameter_names, components):
    """Locate a Patsy interaction term without assuming component order."""
    target = set(components)
    matches = [
        term
        for term in parameter_names
        if set(term.split(":")) == target
        and len(term.split(":")) == len(components)
    ]
    if len(matches) != 1:
        raise KeyError(
            "Expected one interaction term with components "
            f"{components}, found {matches}."
        )
    return matches[0]


intensity_centered = f"{PRIMARY_INTENSITY_COLUMN}_c"
interaction_specs = {
    "settlement_urbanity": {
        "label": "Settlement urbanity",
        "centered_column": "settlement_urbanity_c",
        "interaction_term": (
            f"{intensity_centered}:settlement_urbanity_c"
        ),
    },
    "onsite_network_position": {
        "label": "Existing onsite network position",
        "centered_column": "onsite_network_position_c",
        "interaction_term": (
            f"{intensity_centered}:onsite_network_position_c"
        ),
    },
}


# Complete inference for the two moderation terms.
interaction_inference_rows = []
for role in ["residential", "employment"]:
    for outcome, metadata in CONTEXT_OUTCOMES.items():
        fitted = context_models[(role, outcome)]
        data = context_model_data[(role, outcome)]

        for moderator, specification in interaction_specs.items():
            term = specification["interaction_term"]
            inference = linear_combination_inference(
                fitted,
                {term: 1.0},
            )
            inference.update({
                "role": role,
                "role_label": ROLE_LABELS[role],
                "outcome": outcome,
                "outcome_label": metadata["label"],
                "moderator": moderator,
                "moderator_label": specification["label"],
                "term": term,
                "n_observations": int(fitted.nobs),
                "n_districts": int(
                    data["district_id"].nunique()
                ),
                "covariance_estimator": "HC3",
            })
            interaction_inference_rows.append(inference)

context_interaction_inference = pd.DataFrame(
    interaction_inference_rows
)


# Conditional intensity slopes at P20, P50, and P80.
conditional_slope_rows = []
high_low_contrast_rows = []

for role in ["residential", "employment"]:
    for outcome, metadata in CONTEXT_OUTCOMES.items():
        fitted = context_models[(role, outcome)]
        data = context_model_data[(role, outcome)]

        for moderator, specification in interaction_specs.items():
            centered_column = specification["centered_column"]
            interaction_term = specification["interaction_term"]
            raw_mean = float(data[moderator].mean())
            percentile_values = (
                data[moderator]
                .quantile(CONTEXT_PERCENTILES)
            )

            evaluated_values = {}
            for percentile, raw_value in zip(
                CONTEXT_PERCENTILES,
                percentile_values,
            ):
                raw_value = float(raw_value)
                centered_value = raw_value - raw_mean
                evaluated_values[float(percentile)] = (
                    raw_value,
                    centered_value,
                )

                inference = linear_combination_inference(
                    fitted,
                    {
                        intensity_centered: 1.0,
                        interaction_term: centered_value,
                    },
                )
                inference.update({
                    "role": role,
                    "role_label": ROLE_LABELS[role],
                    "outcome": outcome,
                    "outcome_label": metadata["label"],
                    "moderator": moderator,
                    "moderator_label": specification["label"],
                    "moderator_percentile": float(percentile),
                    "moderator_value": raw_value,
                    "moderator_value_centered": centered_value,
                    "other_moderator_setting": "sample mean",
                    "n_observations": int(fitted.nobs),
                    "n_districts": int(
                        data["district_id"].nunique()
                    ),
                    "covariance_estimator": "HC3",
                })
                for source, target in [
                    (
                        "estimate_log",
                        "effect_per_0_10_intensity_pct",
                    ),
                    (
                        "ci_low_log",
                        "effect_per_0_10_ci_low_pct",
                    ),
                    (
                        "ci_high_log",
                        "effect_per_0_10_ci_high_pct",
                    ),
                ]:
                    inference[target] = (
                        100.0
                        * np.expm1(
                            INTENSITY_EFFECT_INCREMENT
                            * inference[source]
                        )
                    )
                conditional_slope_rows.append(inference)

            low_raw, low_centered = evaluated_values[0.20]
            high_raw, high_centered = evaluated_values[0.80]
            centered_difference = (
                high_centered - low_centered
            )

            contrast = linear_combination_inference(
                fitted,
                {
                    interaction_term: centered_difference,
                },
            )
            contrast.update({
                "role": role,
                "role_label": ROLE_LABELS[role],
                "outcome": outcome,
                "outcome_label": metadata["label"],
                "moderator": moderator,
                "moderator_label": specification["label"],
                "contrast": "P80 minus P20 intensity slope",
                "p20_value": low_raw,
                "p80_value": high_raw,
                "moderator_difference": high_raw - low_raw,
                "n_observations": int(fitted.nobs),
                "n_districts": int(
                    data["district_id"].nunique()
                ),
                "covariance_estimator": "HC3",
            })
            for source, target in [
                (
                    "estimate_log",
                    "contrast_per_0_10_intensity_pct",
                ),
                (
                    "ci_low_log",
                    "contrast_per_0_10_ci_low_pct",
                ),
                (
                    "ci_high_log",
                    "contrast_per_0_10_ci_high_pct",
                ),
            ]:
                contrast[target] = (
                    100.0
                    * np.expm1(
                        INTENSITY_EFFECT_INCREMENT
                        * contrast[source]
                    )
                )
            high_low_contrast_rows.append(contrast)

context_conditional_slopes = pd.DataFrame(
    conditional_slope_rows
)
context_high_low_slope_contrasts = pd.DataFrame(
    high_low_contrast_rows
)


# Formal employment-versus-residential differences in moderation.
# Every outcome uses the matched district sample available in both roles.
cross_role_models = {}
cross_role_test_rows = []

for outcome, metadata in CONTEXT_OUTCOMES.items():
    required_columns = [
        "district_id",
        "role",
        outcome,
        PRIMARY_INTENSITY_COLUMN,
        "settlement_urbanity",
        "onsite_network_position",
        "log_population",
        "log_external_flow",
        "province_code",
    ]
    stacked = (
        period_district[required_columns]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .drop_duplicates(["district_id", "role"])
        .copy()
    )
    matched_districts = (
        stacked.groupby("district_id")["role"]
        .nunique()
        .loc[lambda values: values.eq(2)]
        .index
    )
    stacked = stacked.loc[
        stacked["district_id"].isin(matched_districts)
    ].copy()
    if stacked.empty:
        raise RuntimeError(
            f"No matched residential-employment sample for {outcome}."
        )

    stacked["role_employment"] = (
        stacked["role"].eq("employment").astype(float)
    )
    for column in [
        PRIMARY_INTENSITY_COLUMN,
        "settlement_urbanity",
        "onsite_network_position",
        "log_external_flow",
    ]:
        stacked[f"{column}_c"] = (
            stacked[column] - stacked[column].mean()
        )

    role_comparison_formula = (
        f"{outcome} ~ role_employment * ("
        f"{intensity_centered} * settlement_urbanity_c "
        f"+ {intensity_centered} * onsite_network_position_c "
        "+ log_population "
        "+ log_external_flow_c "
        "+ C(province_code))"
    )
    fitted = smf.ols(
        role_comparison_formula,
        data=stacked,
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": stacked["district_id"],
            "use_correction": True,
        },
    )
    cross_role_models[outcome] = fitted

    test_components = {
        "Intensity slope at mean context": [
            "role_employment",
            intensity_centered,
        ],
        "Intensity-by-urbanity interaction": [
            "role_employment",
            intensity_centered,
            "settlement_urbanity_c",
        ],
        "Intensity-by-onsite-position interaction": [
            "role_employment",
            intensity_centered,
            "onsite_network_position_c",
        ],
    }

    for comparison_label, components in test_components.items():
        term = find_interaction_term(
            fitted.params.index,
            components,
        )
        inference = linear_combination_inference(
            fitted,
            {term: 1.0},
        )
        inference.update({
            "outcome": outcome,
            "outcome_label": metadata["label"],
            "comparison": comparison_label,
            "term": term,
            "difference_direction": (
                "Employment minus residential"
            ),
            "n_observations": int(fitted.nobs),
            "n_districts": int(
                stacked["district_id"].nunique()
            ),
            "covariance_estimator": (
                "district-clustered robust covariance"
            ),
        })
        cross_role_test_rows.append(inference)

context_cross_role_tests = pd.DataFrame(
    cross_role_test_rows
)


# Main and appendix-ready exports.
atomic_to_csv(
    context_interaction_inference,
    TABLE_MAIN_DIR
    / "district_context_interaction_inference.csv",
)
atomic_to_csv(
    context_conditional_slopes,
    TABLE_MAIN_DIR
    / "district_context_conditional_intensity_slopes.csv",
)
atomic_to_csv(
    context_high_low_slope_contrasts,
    TABLE_MAIN_DIR
    / "district_context_high_low_slope_contrasts.csv",
)
atomic_to_csv(
    context_cross_role_tests,
    TABLE_APPENDIX_DIR
    / "district_context_cross_role_interaction_tests.csv",
)

display(
    context_interaction_inference[[
        "role_label",
        "outcome_label",
        "moderator_label",
        "estimate_log",
        "robust_se",
        "ci_low_log",
        "ci_high_log",
        "p_value_label",
        "significance",
    ]].round(4)
)
display(
    context_conditional_slopes[[
        "role_label",
        "outcome_label",
        "moderator_label",
        "moderator_percentile",
        "effect_per_0_10_intensity_pct",
        "effect_per_0_10_ci_low_pct",
        "effect_per_0_10_ci_high_pct",
        "p_value_label",
        "significance",
    ]].round(3)
)
display(
    context_high_low_slope_contrasts[[
        "role_label",
        "outcome_label",
        "moderator_label",
        "contrast_per_0_10_intensity_pct",
        "contrast_per_0_10_ci_low_pct",
        "contrast_per_0_10_ci_high_pct",
        "p_value_label",
        "significance",
    ]].round(3)
)
display(
    context_cross_role_tests[[
        "outcome_label",
        "comparison",
        "estimate_log",
        "robust_se",
        "ci_low_log",
        "ci_high_log",
        "p_value_label",
        "significance",
        "n_districts",
    ]].round(4)
)

## 21. Predicted relationships across settlement urbanity


In [ ]:
urbanity_prediction_parts = []
intensity_centered = f"{PRIMARY_INTENSITY_COLUMN}_c"

for role in ["residential", "employment"]:
    for outcome, metadata in CONTEXT_OUTCOMES.items():
        fitted = context_models[(role, outcome)]
        data = context_model_data[(role, outcome)]

        intensity_grid = np.linspace(
            data[intensity_centered].quantile(0.02),
            data[intensity_centered].quantile(0.98),
            60,
        )

        urbanity_values = data[
            "settlement_urbanity_c"
        ].quantile(CONTEXT_PERCENTILES)

        for percentile, value in zip(
            CONTEXT_PERCENTILES,
            urbanity_values,
        ):
            curve = average_adjusted_curve(
                fitted,
                data,
                x_column=intensity_centered,
                x_values=intensity_grid,
                fixed_values={
                    "settlement_urbanity_c": float(value),
                    "onsite_network_position_c": 0.0,
                },
            )

            offset = metadata["backtransform_offset"]
            if offset:
                for column in [
                    "prediction",
                    "ci_low",
                    "ci_high",
                ]:
                    curve[column] = np.maximum(
                        curve[column] - offset,
                        0.0,
                    )

            curve["hybrid_work_intensity"] = (
                curve[intensity_centered]
                + data[PRIMARY_INTENSITY_COLUMN].mean()
            )
            curve["role"] = role
            curve["outcome"] = outcome
            curve["outcome_label"] = metadata["label"]
            curve["context_percentile"] = percentile
            curve["context_label"] = {
                0.20: "Lower urbanity",
                0.50: "Median urbanity",
                0.80: "Higher urbanity",
            }[round(percentile, 2)]

            urbanity_prediction_parts.append(curve)

urbanity_predictions = pd.concat(
    urbanity_prediction_parts,
    ignore_index=True,
)

atomic_to_parquet(
    urbanity_predictions,
    TABLE_MAIN_DIR
    / "urbanity_intensity_prediction_curves.parquet",
)

figure, axes = plt.subplots(
    2,
    3,
    figsize=(14.5, 8.0),
)

line_styles = {
    "Lower urbanity": "--",
    "Median urbanity": "-",
    "Higher urbanity": ":",
}

for row_index, role in enumerate(
    ["residential", "employment"]
):
    for column_index, (outcome, metadata) in enumerate(
        CONTEXT_OUTCOMES.items()
    ):
        axis = axes[row_index, column_index]

        frame = urbanity_predictions.loc[
            urbanity_predictions["role"].eq(role)
            & urbanity_predictions["outcome"].eq(outcome)
        ]

        for context_label, part in frame.groupby(
            "context_label",
            sort=False,
        ):
            axis.plot(
                part["hybrid_work_intensity"],
                part["prediction"],
                label=context_label,
                linestyle=line_styles[context_label],
                linewidth=1.8,
            )
            axis.fill_between(
                part["hybrid_work_intensity"],
                part["ci_low"],
                part["ci_high"],
                alpha=0.10,
            )

        axis.set_title(metadata["label"])
        axis.set_xlabel("Hybrid-work intensity")
        axis.set_ylabel(
            ROLE_LABELS[role]
            if column_index == 0
            else metadata["label"]
        )
        axis.grid(axis="y", alpha=0.25)

axes[0, 0].legend(
    frameon=False,
    fontsize=8,
)

figure.suptitle(
    "District network outcomes across hybrid-work intensity and settlement urbanity"
)
figure.subplots_adjust(
    top=0.90,
    hspace=0.42,
    wspace=0.30,
)

save_figure(
    figure,
    "figure_5_urbanity_moderation_of_intensity_outcomes",
    FIGURE_MAIN_DIR,
)


## 22. Predicted relationships across existing onsite network position


In [ ]:
position_prediction_parts = []
intensity_centered = f"{PRIMARY_INTENSITY_COLUMN}_c"

for role in ["residential", "employment"]:
    for outcome, metadata in CONTEXT_OUTCOMES.items():
        fitted = context_models[(role, outcome)]
        data = context_model_data[(role, outcome)]

        intensity_grid = np.linspace(
            data[intensity_centered].quantile(0.02),
            data[intensity_centered].quantile(0.98),
            60,
        )

        position_values = data[
            "onsite_network_position_c"
        ].quantile(CONTEXT_PERCENTILES)

        for percentile, value in zip(
            CONTEXT_PERCENTILES,
            position_values,
        ):
            curve = average_adjusted_curve(
                fitted,
                data,
                x_column=intensity_centered,
                x_values=intensity_grid,
                fixed_values={
                    "settlement_urbanity_c": 0.0,
                    "onsite_network_position_c": float(value),
                },
            )

            offset = metadata["backtransform_offset"]
            if offset:
                for column in [
                    "prediction",
                    "ci_low",
                    "ci_high",
                ]:
                    curve[column] = np.maximum(
                        curve[column] - offset,
                        0.0,
                    )

            curve["hybrid_work_intensity"] = (
                curve[intensity_centered]
                + data[PRIMARY_INTENSITY_COLUMN].mean()
            )
            curve["role"] = role
            curve["outcome"] = outcome
            curve["outcome_label"] = metadata["label"]
            curve["context_percentile"] = percentile
            curve["context_label"] = {
                0.20: "Lower onsite position",
                0.50: "Median onsite position",
                0.80: "Higher onsite position",
            }[round(percentile, 2)]

            position_prediction_parts.append(curve)

position_predictions = pd.concat(
    position_prediction_parts,
    ignore_index=True,
)

atomic_to_parquet(
    position_predictions,
    TABLE_MAIN_DIR
    / "network_position_intensity_prediction_curves.parquet",
)

figure, axes = plt.subplots(
    2,
    3,
    figsize=(14.5, 8.0),
)

line_styles = {
    "Lower onsite position": "--",
    "Median onsite position": "-",
    "Higher onsite position": ":",
}

for row_index, role in enumerate(
    ["residential", "employment"]
):
    for column_index, (outcome, metadata) in enumerate(
        CONTEXT_OUTCOMES.items()
    ):
        axis = axes[row_index, column_index]

        frame = position_predictions.loc[
            position_predictions["role"].eq(role)
            & position_predictions["outcome"].eq(outcome)
        ]

        for context_label, part in frame.groupby(
            "context_label",
            sort=False,
        ):
            axis.plot(
                part["hybrid_work_intensity"],
                part["prediction"],
                label=context_label,
                linestyle=line_styles[context_label],
                linewidth=1.8,
            )
            axis.fill_between(
                part["hybrid_work_intensity"],
                part["ci_low"],
                part["ci_high"],
                alpha=0.10,
            )

        axis.set_title(metadata["label"])
        axis.set_xlabel("Hybrid-work intensity")
        axis.set_ylabel(
            ROLE_LABELS[role]
            if column_index == 0
            else metadata["label"]
        )
        axis.grid(axis="y", alpha=0.25)

axes[0, 0].legend(
    frameon=False,
    fontsize=8,
)

figure.suptitle(
    "District network outcomes across hybrid-work intensity and existing onsite position"
)
figure.subplots_adjust(
    top=0.90,
    hspace=0.42,
    wspace=0.30,
)

save_figure(
    figure,
    "figure_6_network_position_moderation_of_intensity_outcomes",
    FIGURE_MAIN_DIR,
)


## 23. District-specific marginal intensity relationships for mapping


In [ ]:
marginal_parts = []

for role in ["residential", "employment"]:
    for outcome, metadata in CONTEXT_OUTCOMES.items():
        fitted = context_models[(role, outcome)]
        data = context_model_data[(role, outcome)]

        effects = intensity_marginal_effects(
            fitted,
            data,
            intensity_column=PRIMARY_INTENSITY_COLUMN,
        )

        keep_columns = [
            "district_id",
            "role",
            PRIMARY_INTENSITY_COLUMN,
            ROBUSTNESS_INTENSITY_COLUMN,
            "settlement_urbanity",
            "onsite_network_position",
            "intensity_slope_log",
            "intensity_slope_se",
            "intensity_slope_ci_low",
            "intensity_slope_ci_high",
            "intensity_effect_per_0_1_pct",
            "intensity_effect_per_0_1_ci_low_pct",
            "intensity_effect_per_0_1_ci_high_pct",
        ]

        keep_columns = [
            column
            for column in keep_columns
            if column in effects.columns
        ]

        effects = effects[
            keep_columns
        ].copy()

        effects["outcome"] = outcome
        effects["outcome_label"] = metadata["label"]
        effects["intensity_measure"] = PRIMARY_INTENSITY_COLUMN

        marginal_parts.append(effects)

context_marginal_effects = pd.concat(
    marginal_parts,
    ignore_index=True,
)

atomic_to_parquet(
    context_marginal_effects,
    TABLE_MAIN_DIR
    / "district_context_marginal_intensity_effects.parquet",
)
atomic_to_parquet(
    context_marginal_effects,
    MAP_INPUT_DIR
    / "district_context_marginal_intensity_effects_map_ready.parquet",
)

display(context_marginal_effects.head())


# Appendix analyses and reproducibility documentation

The main text uses the full six-layer gradient and continuous district intensity. The following outputs document alternative partner-coverage draws, quadratic intensity terms, all-flow intensity, spectral network position, and the earlier adjacent-layer comparison when a reviewer requests it. These checks do not organise the main argument.


## 24. Alternative partner-coverage draws and nonlinear intensity specifications


In [ ]:
robustness_rows = []

for role in ["residential", "employment"]:
    role_data = district_month.loc[
        district_month["role"].eq(role)
        & district_month["valid_intensity"]
    ].copy()

    # Standardised rarefied partner coverage.
    for coverage_column in [
        "partner_coverage_rarefied_q05",
        "partner_coverage_rarefied_q10",
        "partner_coverage_rarefied_q15",
    ]:
        transformed = f"log_{coverage_column}"

        role_data[transformed] = np.where(
            role_data[coverage_column] > 0,
            np.log(role_data[coverage_column]),
            np.nan,
        )

        fitted = fit_two_way_fixed_effect_model(
            role_data,
            outcome=transformed,
            predictors=[
                PRIMARY_INTENSITY_COLUMN,
                "log_external_flow",
            ],
        )

        coefficient = coefficient_table(
            fitted,
            model_name=coverage_column,
        ).loc[
            lambda x: x["term"].eq(
                PRIMARY_INTENSITY_COLUMN
            )
        ].iloc[0].to_dict()

        coefficient.update({
            "role": role,
            "outcome": "partner_coverage",
            "specification": coverage_column,
            "intensity_measure": PRIMARY_INTENSITY_COLUMN,
        })
        robustness_rows.append(coefficient)

    # External-flow intensity as an alternative intensity definition.
    for outcome, metadata in OUTCOME_DEFINITIONS.items():
        fitted_external = fit_two_way_fixed_effect_model(
            role_data,
            outcome=outcome,
            predictors=[
                ROBUSTNESS_INTENSITY_COLUMN,
                "log_external_flow",
            ],
        )

        coefficient = coefficient_table(
            fitted_external,
            model_name="external_intensity",
        ).loc[
            lambda x: x["term"].eq(
                ROBUSTNESS_INTENSITY_COLUMN
            )
        ].iloc[0].to_dict()

        coefficient.update({
            "role": role,
            "outcome": outcome,
            "specification": "external_flow_intensity",
            "intensity_measure": ROBUSTNESS_INTENSITY_COLUMN,
        })
        robustness_rows.append(coefficient)

    # Quadratic primary-intensity specifications.
    for outcome, metadata in OUTCOME_DEFINITIONS.items():
        nonlinear_data = role_data.copy()
        nonlinear_data[
            "hybrid_intensity_squared"
        ] = (
            nonlinear_data[
                PRIMARY_INTENSITY_COLUMN
            ] ** 2
        )

        fitted = fit_two_way_fixed_effect_model(
            nonlinear_data,
            outcome=outcome,
            predictors=[
                PRIMARY_INTENSITY_COLUMN,
                "hybrid_intensity_squared",
                "log_external_flow",
            ],
        )

        for _, coefficient in coefficient_table(
            fitted,
            model_name="quadratic",
        ).iterrows():
            row = coefficient.to_dict()
            row.update({
                "role": role,
                "outcome": outcome,
                "specification": "quadratic_intensity",
                "intensity_measure": PRIMARY_INTENSITY_COLUMN,
            })
            robustness_rows.append(row)

robustness_results = pd.DataFrame(
    robustness_rows
)

atomic_to_csv(
    robustness_results,
    TABLE_APPENDIX_DIR
    / "intensity_model_robustness.csv",
)

display(robustness_results.head(30))


## 25. Panel fixed-effects moderation robustness

This appendix specification tests whether the contextual differences also appear in within-district monthly change. District and month effects are removed, external-flow volume remains controlled, and standard errors are clustered by district. The period interaction models and their existing figures remain the main analysis.

In [ ]:
# ---------------------------------------------------------------------
# Within-district panel moderation robustness
# ---------------------------------------------------------------------
panel_context = district_month.merge(
    onsite_position[[
        "district_id",
        "role",
        "onsite_network_position",
    ]],
    on=["district_id", "role"],
    how="left",
    validate="many_to_one",
)

panel_context_rows = []
panel_context_models = {}

for role in ["residential", "employment"]:
    role_data = panel_context.loc[
        panel_context["role"].eq(role)
        & panel_context["valid_intensity"]
    ].copy()

    for outcome, metadata in CONTEXT_OUTCOMES.items():
        required_columns = [
            outcome,
            PRIMARY_INTENSITY_COLUMN,
            "settlement_urbanity",
            "onsite_network_position",
            "log_external_flow",
            "district_id",
            "month",
        ]
        require_columns(
            role_data,
            required_columns,
            f"panel context input ({role}, {outcome})",
        )
        model_data = (
            role_data
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=required_columns)
            .copy()
        )
        if model_data.empty:
            raise RuntimeError(
                f"No panel context observations for {role}, {outcome}."
            )

        district_context = (
            model_data[[
                "district_id",
                "settlement_urbanity",
                "onsite_network_position",
            ]]
            .drop_duplicates("district_id")
        )
        urbanity_mean = float(
            district_context["settlement_urbanity"].mean()
        )
        position_mean = float(
            district_context["onsite_network_position"].mean()
        )

        model_data["settlement_urbanity_c"] = (
            model_data["settlement_urbanity"]
            - urbanity_mean
        )
        model_data["onsite_network_position_c"] = (
            model_data["onsite_network_position"]
            - position_mean
        )
        model_data["intensity_x_urbanity"] = (
            model_data[PRIMARY_INTENSITY_COLUMN]
            * model_data["settlement_urbanity_c"]
        )
        model_data["intensity_x_onsite_position"] = (
            model_data[PRIMARY_INTENSITY_COLUMN]
            * model_data["onsite_network_position_c"]
        )

        fitted = fit_two_way_fixed_effect_model(
            model_data,
            outcome=outcome,
            predictors=[
                PRIMARY_INTENSITY_COLUMN,
                "intensity_x_urbanity",
                "intensity_x_onsite_position",
                "log_external_flow",
            ],
        )
        panel_context_models[(role, outcome)] = fitted

        table = coefficient_table(
            fitted,
            model_name=(
                f"panel_context_{role}_{outcome}"
            ),
        )
        table = table.loc[
            table["term"].isin([
                PRIMARY_INTENSITY_COLUMN,
                "intensity_x_urbanity",
                "intensity_x_onsite_position",
            ])
        ].copy()
        table["role"] = role
        table["role_label"] = ROLE_LABELS[role]
        table["outcome"] = outcome
        table["outcome_label"] = metadata["label"]
        table["n_districts"] = int(
            fitted._analysis_data["district_id"].nunique()
        )
        table["n_months"] = int(
            fitted._analysis_data["month"].nunique()
        )
        table["demeaning_iterations"] = int(
            fitted._demeaning_iterations
        )
        table["covariance_estimator"] = (
            "district-clustered robust covariance"
        )
        panel_context_rows.extend(
            table.to_dict("records")
        )

panel_context_robustness = pd.DataFrame(
    panel_context_rows
)
panel_context_robustness["p_value_label"] = (
    panel_context_robustness["p_value"]
    .map(context_p_value_label)
)
panel_context_robustness["significance"] = (
    panel_context_robustness["p_value"]
    .map(context_significance_marker)
)

atomic_to_csv(
    panel_context_robustness,
    TABLE_APPENDIX_DIR
    / "district_panel_context_interaction_robustness.csv",
)

display(
    panel_context_robustness[[
        "role_label",
        "outcome_label",
        "term",
        "estimate",
        "robust_se",
        "ci_low",
        "ci_high",
        "p_value_label",
        "significance",
        "n",
        "n_districts",
        "n_months",
    ]].round(4)
)

## 26. Data dictionary, figure inventory, and reproducibility record


In [ ]:
DATA_DICTIONARY = [
    {
        "field": "hybrid_work_intensity",
        "definition": (
            "Layer score from 0 for 14 workplace days "
            "to 1 for 1–2 workplace days."
        ),
    },
    {
        "field": "hybrid_intensity_all_flows",
        "definition": (
            "Primary district-role flow-weighted hybrid-work intensity "
            "across all six layers, including within-district flows."
        ),
    },
    {
        "field": "hybrid_intensity_external",
        "definition": (
            "Robustness district-role hybrid-work intensity calculated "
            "from external flows only."
        ),
    },
    {
        "field": "partner_coverage",
        "definition": (
            "Observed number of external district partners. "
            "External flow is controlled in the formal models."
        ),
    },
    {
        "field": "partner_coverage_rarefied_q10",
        "definition": (
            f"Expected external partners at the 10th-percentile "
            f"common draw of {main_partner_draws} flow units."
        ),
    },
    {
        "field": "partner_diversity",
        "definition": (
            "Exponentiated Shannon entropy of external partner shares; "
            "set to zero where no external relation exists."
        ),
    },
    {
        "field": "mean_external_distance_km",
        "definition": (
            "Flow-weighted mean centroid distance of external district "
            "connections; undefined where no external relation exists."
        ),
    },
    {
        "field": "onsite_network_position",
        "definition": (
            "Percentile rank of residential outgoing or employment incoming "
            "strength in the 8–14-day attendance network. "
            "Districts with no onsite strength receive zero."
        ),
    },
]

atomic_to_csv(
    pd.DataFrame(DATA_DICTIONARY),
    TABLE_APPENDIX_DIR / "data_dictionary.csv",
)

figure_inventory = []

for directory, category in [
    (FIGURE_MAIN_DIR, "main"),
    (FIGURE_APPENDIX_DIR, "appendix"),
]:
    for path in sorted(directory.glob("*")):
        if path.is_file():
            figure_inventory.append({
                "category": category,
                "filename": path.name,
                "path": str(path),
                "size_bytes": path.stat().st_size,
            })

figure_inventory = pd.DataFrame(
    figure_inventory
)

atomic_to_csv(
    figure_inventory,
    LOG_DIR / "figure_inventory.csv",
)

manifest = {
    "run_version": RUN_VERSION,
    "run_date": RUN_DATE,
    "run_tag": RUN_TAG,
    "notebook": (
        "01_Hybrid_Work_Job_Home_Networks_Formal_Analysis_"
        "v4_4_20260724"
    ),
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "analysis_signature": ANALYSIS_SIGNATURE,
    "valid_months": VALID_MONTHS,
    "excluded_months": (
        month_qc.loc[
            ~month_qc["valid_month"],
            "month",
        ]
        .astype(str)
        .sort_values()
        .tolist()
    ),
    "recurrence_band_order": RECURRENCE_BAND_ORDER,
    "recurrence_intensity": RECURRENCE_INTENSITY,
    "primary_intensity_column": PRIMARY_INTENSITY_COLUMN,
    "robustness_intensity_column": ROBUSTNESS_INTENSITY_COLUMN,
    "primary_partner_coverage_definition": (
        PRIMARY_PARTNER_COVERAGE_DEFINITION
    ),
    "sample_strategy": (
        "maximum available observations for each outcome; "
        "no common complete-case outcome restriction"
    ),
    "structural_zero_policy": {
        "partner_coverage": 0,
        "partner_diversity": 0,
        "mean_external_distance": "undefined",
    },
    "global_layer_common_draws": global_layer_common_draws,
    "rarefied_partner_draws": coverage_draws,
    "period_partner_draws": period_partner_draws,
    "main_figure_stems": MAIN_FIGURE_STEMS,
    "stage_dir": str(STAGE_DIR),
    "map_input_dir": str(MAP_INPUT_DIR),
    "ghs_profile_path": str(ghs_path),
    "ghs_profile_origin": ghs_profile_origin,
    "ghsl_release": GHSL_RELEASE,
    "ghsl_epoch": GHSL_EPOCH,
    "ghsl_resolution_metres": GHSL_RESOLUTION_METRES,
    "ghs_urbanity_definition": (
        "Population-weighted mean of ordinal scores assigned to "
        "seven inhabited GHS-SMOD classes"
    ),
}

atomic_write_json(
    manifest,
    LOG_DIR / "analysis_manifest.json",
)

print(json.dumps(manifest, indent=2))
